# In-Context Engram (ICE) — hybrid GDN language models on Kaggle

Implements the proposal *"In-Context Engram: An Attention-Free Exact Copy Path for Hybrid Language Models
with Less KV Cache"*: Gated DeltaNet + full-attention hybrids, the ICE lookup, arms A–F, and the evaluations.

**Kaggle settings:** *Accelerator* → **GPU T4 ×2** (or any GPU), *Internet* → **On**,
*Persistence* → **Files only** (so checkpoints survive between sessions).

**Run order:** 1 Config → 2 Setup → 3 Write code → 4 Tests → 5 Data → 6 Day 0 → 7 Train → 8 Evaluate →
9 Inference cost → 10 Figures.

### Be realistic about compute
| preset | params | tokens/s on 2×T4 (rough) | 12 h session ≈ |
|---|---|---|---|
| `smoke` | 1M | — (seconds) | pipeline check |
| `140m` | ~150M | ~7–20K (fla or not) | ~0.3–0.9B tokens |
| `0.6b` | ~0.6B | ~1.5–5K | ~0.06–0.2B tokens |

Kaggle gives ~30 GPU-hours/week. **Recommended main scale: `140m`** (default recipe: 400M tokens, batch 131K
tokens, lr 1e-3; ~1–2 sessions per arm). `0.6b` (default 300M tokens) is a sanity check at ~0.5 tokens/param.
Runs continue across sessions with **automatic resume** (see the last section); the training log prints the
measured tokens/s and an ETA after the first steps.

**Fair comparison:** every arm with the same `PRESET`, `TOTAL_TOKENS`, `SEQ_LEN`, `SEED`, number of GPUs and
GPU type (the micro-batch is picked from GPU memory), so all arms see the same data in the same order.


## 1. Config

In [ ]:
# ---- edit me ----------------------------------------------------------------
PRESET       = "0.6b"        # "smoke" | "140m" (recommended main scale) | "0.6b" (sanity check, slow on T4)
ARM          = "C"           # A B C F, controls B_pm C_notrust (+ D E C_nodrop C_in C_out)
SEED         = 1234          # paired seeds: use the same seed list for every arm (e.g. 1234, 1235)
TOTAL_TOKENS = None          # None = preset default (140m: 400M, 0.6b: 300M); same value for every arm
TRAIN_DATA_TOKENS = 420_000_000   # tokens to download/tokenize: >= TOTAL_TOKENS so no window repeats (~2 B/token on disk)
MAX_HOURS    = 11.0          # stop + checkpoint before Kaggle's session limit
GRAD_CKPT    = "auto"        # "auto" = on for 16 GB GPUs; try False for 140m (~25% faster) if it fits in memory
SEQ_LEN      = 4096
WORK         = "/kaggle/working"            # outputs (persisted with 'Save Version')
DATA_DIR     = f"{WORK}/data"
RUN_DIR      = f"{WORK}/runs/{PRESET}_{ARM}" + ("" if SEED == 1234 else f"_s{SEED}")
# To resume from a previous session: attach that notebook's output as an input and set this to the
# folder that contains ckpt_latest.pt, e.g. "/kaggle/input/<previous-version>/runs/0.6b_C"
RESUME_FROM  = None
# --------------------------------------------------------------------------------
import os, json, subprocess, time
SESSION_START = time.time()     # MAX_HOURS counts from here (setup + data prep included)
KAGGLE_LIMIT_H = 12.0          # Kaggle ends a session after 12 h
def hours_left():               # until MAX_HOURS: training stops (and checkpoints) here
    return MAX_HOURS - (time.time() - SESSION_START) / 3600
def session_left():             # until Kaggle ends the session
    return KAGGLE_LIMIT_H - (time.time() - SESSION_START) / 3600
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print(PRESET, ARM, SEED, TOTAL_TOKENS, RUN_DIR)

## 2. Setup (GPU check + optional fast kernels)
`flash-linear-attention` gives fast GDN kernels on recent GPUs. If it does not install or does not work on this GPU (common on T4/P100), the code automatically falls back to its exact pure-PyTorch kernel.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU"
import torch
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", N_GPU,
      [torch.cuda.get_device_name(i) for i in range(N_GPU)])
!pip install -q --no-deps fla-core==0.4.2 flash-linear-attention==0.4.2 2>&1 | tail -2 || true
!pip install -q "datasets>=2.19" "transformers>=4.40" 2>&1 | tail -2 || true

## 3. Write the codebase
Each cell below writes one file of the `ice_lm` package.

In [ ]:
!mkdir -p ice_lm/ice_lm ice_lm/scripts ice_lm/tests

In [ ]:
%%writefile ice_lm/ice_lm/__init__.py
"""ice_lm: hybrid GDN / attention language models with In-Context Engram (ICE)."""
from .config import ARMS, PRESETS, ModelConfig, TrainConfig, build_configs  # noqa: F401
from .model import ICELM, kv_bytes_per_token, layer_plan  # noqa: F401


In [ ]:
%%writefile ice_lm/ice_lm/config.py
"""Model / training configs, size presets and experiment arms (A-F + ablations).

Arms follow the proposal (proposal_ice/proposal.tex, Section 2.2):

    A  layer-level 3:1                       no ICE
    B  layer-level 7:1                       no ICE
    C  layer-level 7:1                       ICE (input + output + trust signals, dropout 0.2)
    D  head-level hybrid, KV budget of B     no ICE
    E  head-level hybrid, KV budget of B     ICE
    F  layer-level 15:1                      ICE
    C_in / C_out / C_nodrop / C_notrust      ablations of C
    B_pm                                     B with as many parameters as C (capacity control)
"""
from __future__ import annotations

import dataclasses
import json
import os
from dataclasses import dataclass, field
from typing import Optional, Tuple


@dataclass
class ModelConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_layers: int = 16
    n_heads: int = 12            # total heads per layer (GDN heads + attention query heads)
    head_dim: int = 64
    n_kv_heads: int = 4          # KV heads of a full-attention layer (GQA)
    mlp_hidden: int = 2048
    # hybrid layout
    layout: str = "layer"        # "layer": whole layers are GDN or attention; "head": mixed heads
    ratio: int = 3               # linear:full ratio for layout="layer"
    head_kv_budget_ratio: int = 7  # layout="head": match the KV budget of a layer-level hybrid with this ratio
    conv_size: int = 4
    rope_theta: float = 10000.0
    qk_norm: bool = True
    gdn_backend: str = "auto"    # "auto" | "fla" | "torch"
    gdn_chunk: int = 64
    gdn_compile: bool = False    # torch.compile the pure-PyTorch GDN kernel (only used when fla is not)
    tie_embeddings: bool = True
    grad_checkpointing: bool = False
    init_std: float = 0.02
    # ICE
    ice: bool = False
    ice_input: bool = True
    ice_output: bool = True
    ice_inject_after: int = 2    # inject after this many blocks
    ice_trust: bool = True       # agreement + online-reliability features (when to ignore a lookup)
    ice_orders: Tuple[int, ...] = (1, 2, 3, 4, 6, 8)


@dataclass
class TrainConfig:
    preset: str = "140m"
    arm: str = "A"
    data_dir: str = "data"
    out_dir: str = "runs/A"
    seq_len: int = 4096
    micro_batch: int = 4             # sequences per GPU per micro-step
    global_batch_tokens: int = 2 ** 19   # ~524K tokens per optimizer step
    total_tokens: int = 2_500_000_000
    lr: float = 3e-4
    min_lr_ratio: float = 0.1
    warmup_tokens: int = 50_000_000
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    ice_dropout: float = 0.2
    seed: int = 1234
    precision: str = "auto"          # "auto" | "bf16" | "fp16" | "fp32"
    compile: bool = False
    zero_optimizer: bool = True      # multi-GPU: shard the Adam states over the GPUs (ZeRO-1)
    num_workers: int = 2
    log_every: int = 10
    eval_every: int = 250
    eval_batches: int = 8
    ckpt_every_minutes: float = 30.0
    max_hours: float = 1e9           # stop (and checkpoint) after this wall-clock time
    resume: bool = True
    keep_resume_config: bool = True  # a resumed run keeps its saved schedule / data-order settings
    keep_checkpoint: bool = False    # keep ckpt_latest.pt (+ optimizer shards) after model_final.pt is written


# ---------------------------------------------------------------------------
# Size presets
# ---------------------------------------------------------------------------
PRESETS = {
    # tiny model for CPU tests / smoke runs
    "smoke": dict(
        model=dict(d_model=128, n_layers=4, n_heads=4, head_dim=32, n_kv_heads=2, mlp_hidden=256,
                   gdn_chunk=32),
        train=dict(seq_len=256, micro_batch=8, global_batch_tokens=256 * 8, total_tokens=2_000_000,
                   lr=2e-3, warmup_tokens=100_000, eval_every=50, eval_batches=4, log_every=10,
                   num_workers=0),
    ),
    # main scale: ~150M parameters. Recipe for a ~400M-token budget (2-3 tokens/param, Kaggle 2xT4):
    # batch 131K tokens -> ~3K optimizer steps, lr 1e-3 (QK-norm keeps it stable), 5% warmup, cosine to 10%
    "140m": dict(
        model=dict(d_model=768, n_layers=16, n_heads=12, head_dim=64, n_kv_heads=4, mlp_hidden=2048),
        train=dict(seq_len=4096, micro_batch=4, global_batch_tokens=2 ** 17,
                   total_tokens=400_000_000, lr=1e-3, warmup_tokens=20_000_000),
    ),
    # ~0.6B parameters (with a 32K vocabulary); at ~300M tokens this is a sanity check, not a converged model
    "0.6b": dict(
        model=dict(d_model=1280, n_layers=24, n_heads=10, head_dim=128, n_kv_heads=2, mlp_hidden=3840,
                   grad_checkpointing=True),
        train=dict(seq_len=4096, micro_batch=1, global_batch_tokens=2 ** 18,
                   total_tokens=300_000_000, lr=6e-4, warmup_tokens=20_000_000),
    ),
}

ARMS = {
    "A": dict(layout="layer", ratio=3, ice=False),
    "B": dict(layout="layer", ratio=7, ice=False),
    "C": dict(layout="layer", ratio=7, ice=True),
    "D": dict(layout="head", head_kv_budget_ratio=7, ice=False),
    "E": dict(layout="head", head_kv_budget_ratio=7, ice=True),
    "F": dict(layout="layer", ratio=15, ice=True),
    # ablations of C
    "C_in": dict(layout="layer", ratio=7, ice=True, ice_output=False),
    "C_out": dict(layout="layer", ratio=7, ice=True, ice_input=False),
    "C_nodrop": dict(layout="layer", ratio=7, ice=True, _ice_dropout=0.0),
    "C_notrust": dict(layout="layer", ratio=7, ice=True, ice_trust=False),   # original ICE (no agree / rel)
    # control: B with a wider MLP so it has as many parameters as C (is C's gain just extra capacity?)
    "B_pm": dict(layout="layer", ratio=7, ice=False, _match_params_of="C"),
}


def ice_param_count(m: ModelConfig) -> int:
    """Parameters ICE adds to a model (input module with trust embeddings + output module)."""
    from .ice_features import N_AGREE_BUCKETS, N_DIST_BUCKETS, N_LEN_BUCKETS, N_REL_BUCKETS
    d, n = m.d_model, len(m.ice_orders)
    p_in = n * d * d + (N_LEN_BUCKETS + N_DIST_BUCKETS) * d + 2 * d
    p_out = d + d + N_LEN_BUCKETS
    if m.ice_trust:
        p_in += n * (N_AGREE_BUCKETS + N_REL_BUCKETS) * d
        p_out += N_AGREE_BUCKETS + N_REL_BUCKETS
    return (p_in if m.ice_input else 0) + (p_out if m.ice_output else 0)


def build_configs(preset: str, arm: str, **overrides) -> Tuple[ModelConfig, TrainConfig]:
    """Combine preset + arm + explicit overrides (overrides may target either config)."""
    if preset not in PRESETS:
        raise ValueError(f"unknown preset {preset!r}; choose from {list(PRESETS)}")
    if arm not in ARMS:
        raise ValueError(f"unknown arm {arm!r}; choose from {list(ARMS)}")
    m = ModelConfig(**PRESETS[preset]["model"])
    t = TrainConfig(preset=preset, arm=arm, **PRESETS[preset]["train"])
    arm_cfg = dict(ARMS[arm])
    if "_ice_dropout" in arm_cfg:
        t.ice_dropout = arm_cfg.pop("_ice_dropout")
    match = arm_cfg.pop("_match_params_of", None)
    m = dataclasses.replace(m, **arm_cfg)
    if match is not None:
        # widen the MLP (3 * d_model params per layer per hidden unit) to absorb the other arm's ICE parameters
        other = dataclasses.replace(ModelConfig(**PRESETS[preset]["model"]),
                                    **{k: v for k, v in ARMS[match].items() if not k.startswith("_")})
        per_unit = 3 * m.d_model * m.n_layers
        m = dataclasses.replace(m, mlp_hidden=m.mlp_hidden + -(-ice_param_count(other) // per_unit))
    mfields = {f.name for f in dataclasses.fields(ModelConfig)}
    tfields = {f.name for f in dataclasses.fields(TrainConfig)}
    for k, v in overrides.items():
        if v is None:
            continue
        if k in mfields:
            m = dataclasses.replace(m, **{k: v})
        elif k in tfields:
            t = dataclasses.replace(t, **{k: v})
        else:
            raise ValueError(f"unknown config key {k!r}")
    return m, t


def to_dict(cfg) -> dict:
    return dataclasses.asdict(cfg)


def model_config_from_dict(d: dict) -> ModelConfig:
    d = dict(d)
    d.setdefault("ice_trust", False)      # runs saved before the trust features existed
    if "ice_orders" in d:
        d["ice_orders"] = tuple(d["ice_orders"])
    return ModelConfig(**d)


def dump_json(path: str, obj) -> None:
    """Atomic write: other processes (DDP ranks) may read the file at the same time."""
    tmp = f"{path}.tmp{os.getpid()}"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


In [ ]:
%%writefile ice_lm/ice_lm/ice_features.py
"""In-Context Engram (ICE) features: an exact, non-learned n-gram lookup over the current context.

For every position t (which predicts token t+1) and every order n, find the most recent earlier
position p < t whose last n tokens equal the last n tokens at t, and report:

    succ[n, t]  = x[p + 1]          the successor token (-1 if no match)
    len_b[n, t] = bucket(true match length at p, extended backwards up to MAX_MATCH)  (0 = no match)
    dist_b[n, t]= bucket(log2(t - p))                                                (0 = no match)

plus the longest-match successor s_star[t] and its length bucket len_star[t].
p < t, so p + 1 <= t: ICE never sees the future.

Two signals tell the model when NOT to trust a lookup (both causal, both cheap):

    agree[n, t] = 0 no match | 1 only one earlier occurrence | 2 the two most recent earlier occurrences
                  had the same successor | 3 they had different successors (ambiguous key)
    rel[n, t]   = reliability of order n in this document so far: of its last REL_WINDOW predictions
                  (positions t' < t with a match), how many were right (succ[n, t'] == x[t'+1]).
                  0 no history | 1 <20% | 2 20-50% | 3 50-80% | 4 >=80%

and the same two for the longest-match prediction s* (agree_star, rel_star).

Documents packed into one window are separated by `seg` (segment ids); matches never cross a
segment boundary. Everything is vectorized NumPy (sort-based), cheap enough for DataLoader workers.
"""
from __future__ import annotations

from typing import Dict, Optional, Sequence

import numpy as np

DEFAULT_ORDERS = (1, 2, 3, 4, 6, 8)
MAX_MATCH = 32
LEN_EDGES = np.array([1, 2, 3, 4, 6, 8, 12, 16, 24, 32])
N_LEN_BUCKETS = len(LEN_EDGES) + 1      # 0 = no match, 1..10
N_DIST_BUCKETS = 18                     # 0 = no match, 1..17 = floor(log2(dist)) + 1 (clipped)
N_AGREE_BUCKETS = 4                     # 0 none, 1 single occurrence, 2 agree, 3 disagree
N_REL_BUCKETS = 5                       # 0 no history, then <20%, 20-50%, 50-80%, >=80% right
REL_WINDOW = 32
REL_EDGES = np.array([0.2, 0.5, 0.8])

_P = np.uint64(0x9E3779B97F4A7C15)


def _len_bucket(length: np.ndarray) -> np.ndarray:
    """0 for length 0, else 1 + index of the largest edge <= length."""
    b = np.searchsorted(LEN_EDGES, length, side="right")  # 0..10, and 0 only if length < 1
    return b.astype(np.int64)


def _dist_bucket(dist: np.ndarray) -> np.ndarray:
    out = np.zeros_like(dist, dtype=np.int64)
    pos = dist > 0
    out[pos] = np.minimum(np.floor(np.log2(dist[pos])).astype(np.int64) + 1, N_DIST_BUCKETS - 1)
    return out


def _reliability(pred: np.ndarray, x: np.ndarray, seg: np.ndarray, window: int = REL_WINDOW) -> np.ndarray:
    """pred [T] (-1 = no prediction). For each t: bucket of the hit rate of the last `window` predictions made
    at positions t' < t of the same segment (a prediction at t' is about x[t'+1], known at time t'+1 <= t)."""
    T = len(pred)
    ev = np.nonzero(pred[: T - 1] >= 0)[0]                       # predictions whose outcome is in the window
    out = np.zeros(T, dtype=np.int64)
    if len(ev) == 0:
        return out
    hits = np.concatenate([[0], np.cumsum(pred[ev] == x[ev + 1])])
    t_idx = np.arange(T)
    seg_start = np.searchsorted(seg, seg, side="left")           # seg is non-decreasing
    hi = np.searchsorted(ev, t_idx, side="left")                 # events with t' < t
    lo = np.maximum(hi - window, np.searchsorted(ev, seg_start, side="left"))
    cnt = hi - lo
    has = cnt > 0
    rate = np.zeros(T)
    rate[has] = (hits[hi[has]] - hits[lo[has]]) / cnt[has]
    out[has] = 1 + np.searchsorted(REL_EDGES, rate[has], side="right")
    return out


def _prev_occurrence(h: np.ndarray, valid: np.ndarray) -> np.ndarray:
    """For each t: most recent p < t with valid[p] and h[p] == h[t] (and valid[t]); else -1."""
    T = len(h)
    prev = np.full(T, -1, dtype=np.int64)
    idx = np.nonzero(valid)[0]
    if len(idx) < 2:
        return prev
    hv = h[idx]
    order = np.argsort(hv, kind="stable")      # stable: equal hashes stay in position order
    hs = hv[order]
    same = hs[1:] == hs[:-1]
    prev[idx[order[1:][same]]] = idx[order[:-1][same]]
    return prev


def compute_ice_features(
    x: np.ndarray,
    seg: Optional[np.ndarray] = None,
    orders: Sequence[int] = DEFAULT_ORDERS,
) -> Dict[str, np.ndarray]:
    """x: [T] token ids. seg: [T] segment (document) ids, non-decreasing; None = one document."""
    x = np.asarray(x, dtype=np.int64)
    T = len(x)
    if seg is None:
        seg = np.zeros(T, dtype=np.int64)
    seg = np.asarray(seg, dtype=np.int64)
    N = len(orders)
    t_idx = np.arange(T)
    xu = x.astype(np.uint64) + np.uint64(1)
    su = seg.astype(np.uint64) + np.uint64(1)

    succ = np.full((N, T), -1, dtype=np.int64)
    len_raw = np.zeros((N, T), dtype=np.int64)
    dist = np.zeros((N, T), dtype=np.int64)
    agree = np.zeros((N, T), dtype=np.int64)

    for j, n in enumerate(orders):
        if n > T:
            continue
        # rolling polynomial hash of x[t-n+1 .. t], mixed with the segment id
        h = np.zeros(T, dtype=np.uint64)
        for i in range(n):
            sh = np.zeros(T, dtype=np.uint64)
            sh[i:] = xu[: T - i]
            h = h * _P + sh
        h = h * _P + su
        valid = np.zeros(T, dtype=bool)
        valid[n - 1:] = seg[n - 1:] == seg[: T - n + 1]   # window inside one document
        p = _prev_occurrence(h, valid)

        has = p >= 0
        if not has.any():
            continue
        # guard against hash collisions: verify the n tokens exactly
        tt, pp = t_idx[has], p[has]
        ok = np.ones(len(tt), dtype=bool)
        for i in range(n):
            ok &= x[tt - i] == x[pp - i]
        has_idx = tt[ok]
        p_ok = pp[ok]

        # true match length: extend backwards up to MAX_MATCH, staying inside the document
        length = np.zeros(len(has_idx), dtype=np.int64)
        alive = np.ones(len(has_idx), dtype=bool)
        for k in range(MAX_MATCH):
            pk = p_ok - k
            tk = has_idx - k
            cand = alive & (pk >= 0)
            m = np.zeros_like(cand)
            ci = np.nonzero(cand)[0]
            m[ci] = (x[tk[ci]] == x[pk[ci]]) & (seg[pk[ci]] == seg[has_idx[ci]])
            length += m
            alive = m
        succ[j, has_idx] = x[p_ok + 1]
        len_raw[j, has_idx] = length
        dist[j, has_idx] = has_idx - p_ok
        # agreement with the occurrence before that one (same n-gram => p's own previous occurrence)
        p2 = p[p_ok]
        ok2 = p2 >= 0
        for i in range(n):                                        # collision guard, as above
            ok2 &= x[np.where(ok2, p_ok - i, 0)] == x[np.where(ok2, p2 - i, 0)]
        a = np.ones(len(has_idx), dtype=np.int64)
        a[ok2] = np.where(x[p2[ok2] + 1] == x[p_ok[ok2] + 1], 2, 3)
        agree[j, has_idx] = a

    len_b = _len_bucket(len_raw)
    dist_b = _dist_bucket(dist)

    # longest match: max true length; ties -> higher order
    score = len_raw * (N + 1) + np.arange(N)[:, None]
    score = np.where(len_raw > 0, score, -1)
    best = np.argmax(score, axis=0)
    match_len = len_raw[best, t_idx]
    s_star = np.where(match_len > 0, succ[best, t_idx], -1)
    len_star = _len_bucket(match_len)
    agree_star = np.where(match_len > 0, agree[best, t_idx], 0)
    dist_star = np.where(match_len > 0, dist[best, t_idx], 0)

    rel = np.stack([_reliability(succ[j], x, seg) for j in range(N)]) if N else np.zeros((0, T), np.int64)
    rel_star = _reliability(s_star, x, seg)

    return dict(
        ice_succ=succ, ice_len=len_b, ice_dist=dist_b, ice_agree=agree, ice_rel=rel,
        ice_s_star=s_star, ice_len_star=len_star, ice_agree_star=agree_star, ice_rel_star=rel_star,
        ice_match_len=match_len, ice_dist_star=dist_star,
    )


def segments_from_eos(x: np.ndarray, eos_id: Optional[int]) -> np.ndarray:
    """Segment ids for a packed window: a new document starts after each EOS token."""
    if eos_id is None:
        return np.zeros(len(x), dtype=np.int64)
    is_eos = (np.asarray(x) == eos_id).astype(np.int64)
    seg = np.concatenate([[0], np.cumsum(is_eos)[:-1]])
    return seg


def brute_force_successor(x: Sequence[int], n: int, seg: Optional[Sequence[int]] = None) -> np.ndarray:
    """Slow reference used by the tests."""
    x = list(x)
    T = len(x)
    seg = [0] * T if seg is None else list(seg)
    out = np.full(T, -1, dtype=np.int64)
    for t in range(n - 1, T):
        if seg[t - n + 1] != seg[t]:
            continue
        for p in range(t - 1, n - 2, -1):
            if seg[p - n + 1] == seg[p] == seg[t] and x[p - n + 1: p + 1] == x[t - n + 1: t + 1]:
                out[t] = x[p + 1]
                break
    return out


In [ ]:
%%writefile ice_lm/ice_lm/gdn.py
"""Gated DeltaNet (GDN) heads with two interchangeable kernels.

Recurrence (per head, state S in R^{K x V}):
    S_t = exp(g_t) * S_{t-1} + beta_t * k_t (v_t - exp(g_t) * S_{t-1}^T k_t)^T
    o_t = S_t^T (scale * q_t)

Kernels
  * "fla"   : flash-linear-attention's Triton chunk kernel (fast; needs a recent GPU + Triton).
  * "torch" : pure-PyTorch chunked (WY / UT-transform) implementation. Works everywhere
              (CPU, T4, P100, ...). Slower, but exact.
"auto" runs a numerical self-test of fla against the torch reference on the actual device / dtype
and silently falls back to torch if fla is missing, broken, or disagrees.
"""
from __future__ import annotations

import inspect
import math
import warnings
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# Kernels
# ---------------------------------------------------------------------------
def gated_delta_rule_recurrent(q, k, v, g, beta, scale: float):
    """Naive step-by-step reference. q,k: [B,T,H,K], v: [B,T,H,V], g,beta: [B,T,H]."""
    B, T, H, K = q.shape
    V = v.shape[-1]
    q, k, v, g, beta = (t.float() for t in (q, k, v, g, beta))
    S = q.new_zeros(B, H, K, V)
    out = []
    for t in range(T):
        S = S * g[:, t].exp()[..., None, None]
        kt, vt = k[:, t], v[:, t]                                   # [B,H,K], [B,H,V]
        pred = torch.einsum("bhk,bhkv->bhv", kt, S)
        S = S + beta[:, t][..., None, None] * torch.einsum("bhk,bhv->bhkv", kt, vt - pred)
        out.append(torch.einsum("bhk,bhkv->bhv", q[:, t] * scale, S))
    return torch.stack(out, dim=1)


def gated_delta_rule_chunked(q, k, v, g, beta, scale: float, chunk_size: int = 64,
                             initial_state: Optional[torch.Tensor] = None, return_state: bool = False):
    """Chunked WY-form implementation (exact). Same shapes as the recurrent reference.
    Always computes in fp32 (autocast disabled), returns the input dtype.
    initial_state / returned state: [B,H,K,V] fp32 (used by the decode cache)."""
    global _COMPILE_TORCH_KERNEL
    with torch.autocast(device_type=q.device.type, enabled=False):
        if _COMPILE_TORCH_KERNEL and initial_state is None and torch.is_grad_enabled():   # training only
            try:
                o, S = _compiled_chunked()(q, k, v, g, beta, scale, chunk_size, None)
                return (o, S) if return_state else o
            except Exception as e:  # noqa: BLE001 - e.g. no compiler toolchain: fall back to eager
                warnings.warn(f"[gdn] torch.compile of the PyTorch kernel failed ({type(e).__name__}); using eager")
                _COMPILE_TORCH_KERNEL = False
        o, S = _chunked_fp32(q, k, v, g, beta, scale, chunk_size, initial_state)
    return (o, S) if return_state else o


_COMPILE_TORCH_KERNEL = False
_COMPILED = None


def set_torch_kernel_compile(on: bool):
    """Opt-in: torch.compile the pure-PyTorch chunked kernel (fewer kernel launches / fused elementwise ops; may help
    on GPUs where fla is unavailable, e.g. T4). Off by default: measure tok/s before relying on it."""
    global _COMPILE_TORCH_KERNEL
    _COMPILE_TORCH_KERNEL = bool(on)


def _compiled_chunked():
    global _COMPILED
    if _COMPILED is None:
        _COMPILED = torch.compile(_chunked_fp32, dynamic=False)
    return _COMPILED


def gated_delta_rule_step(q, k, v, g, beta, scale: float, S: torch.Tensor):
    """One decode step of the recurrence. q,k: [B,H,K], v: [B,H,V], g,beta: [B,H], S: [B,H,K,V] fp32.
    Returns (o [B,H,V] fp32, new S)."""
    q, k, v, g, beta = (t.float() for t in (q, k, v, g, beta))
    S = S * g.exp()[..., None, None]
    pred = torch.einsum("bhk,bhkv->bhv", k, S)
    S = S + beta[..., None, None] * torch.einsum("bhk,bhv->bhkv", k, v - pred)
    return torch.einsum("bhk,bhkv->bhv", q * scale, S), S


def _chunked_fp32(q, k, v, g, beta, scale: float, chunk_size: int, initial_state=None):
    in_dtype = q.dtype
    B, T, H, K = q.shape
    V = v.shape[-1]
    C = chunk_size
    q, k, v, g, beta = (t.float() for t in (q, k, v, g, beta))
    pad = (-T) % C
    if pad:
        q = F.pad(q, (0, 0, 0, 0, 0, pad))
        k = F.pad(k, (0, 0, 0, 0, 0, pad))
        v = F.pad(v, (0, 0, 0, 0, 0, pad))
        g = F.pad(g, (0, 0, 0, pad))
        beta = F.pad(beta, (0, 0, 0, pad))
    Tp = T + pad
    N = Tp // C

    def to_chunks(x):  # [B,Tp,H,*] -> [B,H,N,C,*]
        x = x.transpose(1, 2)
        return x.reshape(B, H, N, C, *x.shape[3:])

    q, k, v = to_chunks(q) * scale, to_chunks(k), to_chunks(v)
    g, beta = to_chunks(g), to_chunks(beta)
    gc = g.cumsum(-1)                                                # [B,H,N,C]

    idx = torch.arange(C, device=q.device)
    lower = idx[:, None] >= idx[None, :]                             # incl. diagonal
    strict = idx[:, None] > idx[None, :]
    diff = gc[..., :, None] - gc[..., None, :]
    gam = torch.where(lower, diff, torch.full_like(diff, -float("inf"))).exp()   # decay i<-j

    kb = k * beta[..., None]
    A = (kb @ k.transpose(-1, -2)) * gam
    A = A.masked_fill(~strict, 0.0)
    eye = torch.eye(C, device=q.device, dtype=q.dtype)
    Tinv = torch.linalg.solve_triangular(eye + A, eye.expand_as(A), upper=False)   # (I + A)^{-1}
    U = Tinv @ (v * beta[..., None])                                 # [B,H,N,C,V]
    W = Tinv @ (kb * gc.exp()[..., None])                            # [B,H,N,C,K]

    S = q.new_zeros(B, H, K, V) if initial_state is None else initial_state.float()
    outs = []
    for n in range(N):
        qn, kn, gcn = q[:, :, n], k[:, :, n], gc[:, :, n]
        v_new = U[:, :, n] - W[:, :, n] @ S
        attn = (qn @ kn.transpose(-1, -2)) * gam[:, :, n]
        outs.append((qn * gcn.exp()[..., None]) @ S + attn @ v_new)
        decay_to_end = (gcn[..., -1:] - gcn).exp()[..., None]       # [B,H,C,1]
        S = S * gcn[..., -1].exp()[..., None, None] + (kn * decay_to_end).transpose(-1, -2) @ v_new
    o = torch.stack(outs, dim=2).reshape(B, H, Tp, V)[:, :, :T]
    return o.transpose(1, 2).to(in_dtype), S      # padded steps have k=v=beta=g=0: S is the state after T


_FLA_FN = None
_FLA_KW: dict = {}


def _fla_call(q, k, v, g, beta, scale, initial_state=None, return_state: bool = False):
    global _FLA_FN, _FLA_KW
    if _FLA_FN is None:
        from fla.ops.gated_delta_rule import chunk_gated_delta_rule
        _FLA_FN = chunk_gated_delta_rule
        # old fla defaulted to head_first=True ([B,H,T,D]); newer fla removed the argument and raises if it is
        # passed (layout is always [B,T,H,D]). The self-test catches a wrong layout either way.
        try:
            p = inspect.signature(inspect.unwrap(_FLA_FN)).parameters.get("head_first")
        except (TypeError, ValueError):
            p = None
        _FLA_KW = dict(head_first=False) if (p is not None and p.default is True) else {}
    o, S = _FLA_FN(q, k, v, g, beta, scale=scale, initial_state=initial_state, output_final_state=return_state,
                   **_FLA_KW)
    return (o, S) if return_state else o


_BACKEND_CACHE = {}


def _fla_self_test(device, dtype, head_dim):
    gen = torch.Generator(device="cpu").manual_seed(0)
    B, T, H, D = 2, 200, 2, head_dim
    q = F.normalize(torch.randn(B, T, H, D, generator=gen), dim=-1).to(device)
    k = F.normalize(torch.randn(B, T, H, D, generator=gen), dim=-1).to(device)
    v = torch.randn(B, T, H, D, generator=gen).to(device)
    g = (-torch.rand(B, T, H, generator=gen) * 0.2).to(device)
    beta = torch.rand(B, T, H, generator=gen).to(device)
    scale = D ** -0.5
    ref = gated_delta_rule_chunked(q, k, v, g, beta, scale)
    qd, kd, vd = (t.to(dtype).requires_grad_() for t in (q, k, v))
    out = _fla_call(qd, kd, vd, g.float(), beta.to(dtype), scale)
    err = ((out.float() - ref).abs().max() / ref.abs().max().clamp_min(1e-6)).item()
    out.float().sum().backward()
    grads_ok = all(t.grad is not None and bool(torch.isfinite(t.grad).all()) for t in (qd, kd, vd))
    tol = 1e-3 if dtype == torch.float32 else 3e-2
    # the inference cache (chunked prefill) passes the state between calls: check a split run too
    state_ok = False
    try:
        with torch.no_grad():
            h = T // 2
            a = [t.to(dtype) for t in (q, k, v)]
            b = beta.to(dtype)
            o1, S1 = _fla_call(a[0][:, :h], a[1][:, :h], a[2][:, :h], g[:, :h].float(), b[:, :h], scale,
                               return_state=True)
            o2, _ = _fla_call(a[0][:, h:], a[1][:, h:], a[2][:, h:], g[:, h:].float(), b[:, h:], scale,
                              initial_state=S1, return_state=True)
            o12 = torch.cat([o1, o2], dim=1).float()
            state_ok = ((o12 - ref).abs().max() / ref.abs().max().clamp_min(1e-6)).item() < tol
    except Exception:  # noqa: BLE001
        state_ok = False
    _FLA_STATE_OK[(str(_norm_device(device)), dtype, head_dim)] = state_ok
    return (err < tol and grads_ok), f"rel_err={err:.2e}, grads_ok={grads_ok}, state_ok={state_ok}"


_FLA_STATE_OK = {}


def fla_state_ok(device, dtype, head_dim) -> bool:
    """Whether fla also passed the split (initial_state / final state) self-test; else the cache path uses torch."""
    return _FLA_STATE_OK.get((str(_norm_device(device)), dtype, head_dim), False)


def _norm_device(device) -> torch.device:
    """"cuda" and "cuda:0" must share one cache entry (prepare_backend gets the former, forward sees the latter)."""
    device = torch.device(device)
    if device.type == "cuda" and device.index is None:
        device = torch.device("cuda", torch.cuda.current_device())
    return device


def resolve_backend(requested: str, device: torch.device, dtype: torch.dtype, head_dim: int) -> str:
    device = _norm_device(device)
    if requested == "torch" or device.type != "cuda":
        if requested == "fla" and device.type != "cuda":
            raise RuntimeError("gdn_backend='fla' needs a CUDA device")
        return "torch"
    key = (requested, str(device), dtype, head_dim)
    if key in _BACKEND_CACHE:
        return _BACKEND_CACHE[key]
    ok, reason = False, ""
    try:
        # the self-test checks gradients too: it must work even if first reached inside torch.no_grad()
        with torch.autocast(device_type="cuda", enabled=False), torch.enable_grad():
            ok, reason = _fla_self_test(device, dtype, head_dim)
    except Exception as e:  # noqa: BLE001 - any failure means "do not use fla"
        reason = f"{type(e).__name__}: {e}"
    if requested == "fla" and not ok:
        raise RuntimeError(f"fla backend requested but self-test failed ({reason})")
    backend = "fla" if ok else "torch"
    if requested == "auto":
        msg = f"[gdn] backend=auto -> {backend} ({reason})"
        print(msg) if ok else warnings.warn(msg + " ; using the pure-PyTorch kernel")
    _BACKEND_CACHE[key] = backend
    return backend


# ---------------------------------------------------------------------------
# GDN heads module
# ---------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class ShortConv(nn.Module):
    """Depthwise causal conv1d + SiLU over [B,T,C]."""

    def __init__(self, channels: int, kernel: int):
        super().__init__()
        self.kernel = kernel
        self.conv = nn.Conv1d(channels, channels, kernel, groups=channels, padding=kernel - 1, bias=False)

    def forward(self, x):
        T = x.shape[1]
        y = self.conv(x.transpose(1, 2))[..., :T]
        return F.silu(y.transpose(1, 2))

    def step(self, x, state):
        """Cached version for prefill / decode. state: [B, kernel-1, C] = the last pre-conv inputs (zeros at the
        start, which reproduces the left zero-padding of forward). Returns (output, new state)."""
        xin = torch.cat([state.to(x.dtype), x], dim=1)                # [B, kernel-1+T, C]
        y = F.conv1d(xin.transpose(1, 2), self.conv.weight.to(x.dtype), groups=xin.shape[-1])
        return F.silu(y.transpose(1, 2)), xin[:, xin.shape[1] - (self.kernel - 1):]


class GatedDeltaNetHeads(nn.Module):
    """n_heads GDN heads of size head_dim. Returns [B, T, n_heads * head_dim] (before o_proj)."""

    def __init__(self, d_model: int, n_heads: int, head_dim: int, conv_size: int = 4,
                 backend: str = "auto", chunk_size: int = 64):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, head_dim
        self.backend, self.chunk_size = backend, chunk_size
        inner = n_heads * head_dim
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.a_proj = nn.Linear(d_model, n_heads, bias=False)     # decay
        self.b_proj = nn.Linear(d_model, n_heads, bias=False)     # write strength beta
        self.g_proj = nn.Linear(d_model, inner, bias=False)       # output gate
        self.conv = ShortConv(3 * inner, conv_size)
        A = torch.empty(n_heads).uniform_(1, 16)
        self.A_log = nn.Parameter(torch.log(A))
        dt = torch.exp(torch.rand(n_heads) * (math.log(0.1) - math.log(0.001)) + math.log(0.001)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))   # inverse softplus
        self.A_log._no_weight_decay = True
        self.dt_bias._no_weight_decay = True
        self.o_norm = RMSNorm(head_dim)

    def forward(self, x, cache=None):
        """cache (inference only): object with .conv [B, conv-1, 3*inner] and .S [B,H,D,D] fp32, updated in place."""
        B, T, _ = x.shape
        H, D = self.n_heads, self.head_dim
        proj = torch.cat([self.q_proj(x), self.k_proj(x), self.v_proj(x)], dim=-1)
        if cache is None:
            qkv = self.conv(proj)
        else:
            qkv, cache.conv = self.conv.step(proj, cache.conv)
        q, k, v = qkv.view(B, T, 3, H, D).unbind(2)
        q = F.normalize(q.float(), dim=-1).to(v.dtype)
        k = F.normalize(k.float(), dim=-1).to(v.dtype)
        beta = torch.sigmoid(self.b_proj(x).float())
        g = -self.A_log.float().exp() * F.softplus(self.a_proj(x).float() + self.dt_bias.float())
        scale = D ** -0.5
        backend = resolve_backend(self.backend, x.device, v.dtype, D)
        if cache is not None and T == 1:                              # decode: one recurrent step
            o, cache.S = gated_delta_rule_step(q[:, 0], k[:, 0], v[:, 0], g[:, 0], beta[:, 0], scale, cache.S)
            o = o[:, None]
        elif backend == "fla" and (cache is None or fla_state_ok(x.device, v.dtype, D)):
            args = (q.contiguous(), k.contiguous(), v.contiguous(), g.contiguous(), beta.to(v.dtype).contiguous(),
                    scale)
            if cache is None:
                o = _fla_call(*args)
            else:
                o, cache.S = _fla_call(*args, initial_state=cache.S, return_state=True)
        elif cache is None:
            o = gated_delta_rule_chunked(q, k, v, g, beta, scale, self.chunk_size)
        else:
            o, cache.S = gated_delta_rule_chunked(q, k, v, g, beta, scale, self.chunk_size, cache.S, True)
        o = self.o_norm(o.to(x.dtype)) * F.silu(self.g_proj(x)).view(B, T, H, D)
        return o.reshape(B, T, H * D)


In [ ]:
%%writefile ice_lm/ice_lm/model.py
"""Hybrid GDN / full-attention language model with optional In-Context Engram (ICE).

Every layer is a HybridMixer with a head allocation:
    * GDN layer:            n_gdn = n_heads, n_attn = 0
    * full-attention layer: n_gdn = 0,       n_attn = n_heads (GQA with n_kv_heads)
    * head-level layer:     both (attention heads are one GQA group: n_heads // n_kv_heads queries, 1 KV head)
ICE (see proposal_ice/ice_explained.pdf):
    input side  (after block `ice_inject_after`):  h <- h + k * sum_n m_n g_n f_n
                f_n = W_n E[s_n] + L[len_n] + D[dist_n] (+ A_n[agree_n] + R_n[rel_n]),
                g_n = sigmoid(<RMSNorm h, RMSNorm f_n> / sqrt(d))
    output side (logits):  z[s*] += k * softplus(w^T RMSNorm(h_last) + b[len*] (+ a[agree*] + r[rel*]))
    agree / rel (ice_features.py) say whether earlier occurrences disagree and how often this order has been
    right so far in the document: they let the model learn when to ignore a misleading lookup.
    k = 1 normally, k = 0 for ICE-dropped sequences (and k = 1 always at test time).
"""
from __future__ import annotations

import math
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from .config import ModelConfig
from .gdn import GatedDeltaNetHeads, RMSNorm, resolve_backend, set_torch_kernel_compile
from .ice_features import N_AGREE_BUCKETS, N_DIST_BUCKETS, N_LEN_BUCKETS, N_REL_BUCKETS

OUT_KEYS = ("ice_s_star", "ice_len_star", "ice_agree_star", "ice_rel_star")


# ---------------------------------------------------------------------------
# Layout: which heads are GDN / attention in each layer
# ---------------------------------------------------------------------------
def _evenly_spaced(n_layers: int, count: int) -> List[int]:
    count = max(1, min(count, n_layers))
    return sorted({round((j + 1) * n_layers / count) - 1 for j in range(count)})


def layer_plan(cfg: ModelConfig) -> List[Dict[str, int]]:
    """Per layer: dict(gdn=#GDN heads, attn=#attention query heads, kv=#KV heads)."""
    L, H, KV = cfg.n_layers, cfg.n_heads, cfg.n_kv_heads
    plan = [dict(gdn=H, attn=0, kv=0) for _ in range(L)]
    if cfg.layout == "layer":
        n_attn_layers = max(1, round(L / (cfg.ratio + 1)))
        for i in _evenly_spaced(L, n_attn_layers):
            plan[i] = dict(gdn=0, attn=H, kv=KV)
    elif cfg.layout == "head":
        # match the KV budget of a layer-level hybrid with ratio head_kv_budget_ratio
        n_ref_layers = max(1, round(L / (cfg.head_kv_budget_ratio + 1)))
        kv_budget = n_ref_layers * KV                     # total KV heads in the reference model
        group = H // KV                                    # query heads per KV head
        for i in _evenly_spaced(L, kv_budget):
            plan[i] = dict(gdn=H - group, attn=group, kv=1)
        assigned = sum(p["kv"] for p in plan)
        if assigned != kv_budget:
            raise ValueError(f"head layout: cannot place {kv_budget} KV heads in {L} layers")
    else:
        raise ValueError(f"unknown layout {cfg.layout!r}")
    return plan


def kv_bytes_per_token(cfg: ModelConfig, bytes_per_el: int = 2) -> int:
    return sum(p["kv"] for p in layer_plan(cfg)) * cfg.head_dim * 2 * bytes_per_el


# ---------------------------------------------------------------------------
# Attention heads
# ---------------------------------------------------------------------------
class Rotary(nn.Module):
    def __init__(self, dim: int, theta: float):
        super().__init__()
        # a plain attribute, not a buffer: model.to(fp16) must not round the frequencies (wrong angles at 32K)
        self.inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=torch.float64) / dim)).float()
        self._cache: Tuple[int, Optional[torch.Tensor], Optional[torch.Tensor]] = (0, None, None)

    def forward(self, T: int, device, offset: int = 0):
        n, cos, sin = self._cache
        end = offset + T
        if n < end or cos is None or cos.device != device:
            t = torch.arange(max(2 * end, 1024), device=device, dtype=torch.float32)   # grow geometrically
            freqs = torch.outer(t, self.inv_freq.to(device))
            cos, sin = freqs.cos(), freqs.sin()
            self._cache = (cos.shape[0], cos, sin)
        return cos[offset:end], sin[offset:end]


def apply_rope(x, cos, sin):  # x: [B,T,H,D]
    x1, x2 = x.float().chunk(2, dim=-1)
    c, s = cos[None, :, None, :], sin[None, :, None, :]
    return torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)


class AttentionHeads(nn.Module):
    def __init__(self, d_model: int, n_q: int, n_kv: int, head_dim: int, rope_theta: float, qk_norm: bool):
        super().__init__()
        self.n_q, self.n_kv, self.head_dim = n_q, n_kv, head_dim
        self.q_proj = nn.Linear(d_model, n_q * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv * head_dim, bias=False)
        self.q_norm = RMSNorm(head_dim) if qk_norm else nn.Identity()
        self.k_norm = RMSNorm(head_dim) if qk_norm else nn.Identity()
        self.rope = Rotary(head_dim, rope_theta)

    def forward(self, x, cache=None, pos: int = 0):
        """cache (inference only): object with preallocated .k/.v [B, n_kv, max_len, D]; x holds positions
        pos .. pos+T-1 and is written into the cache in place."""
        B, T, _ = x.shape
        q = self.q_norm(self.q_proj(x).view(B, T, self.n_q, self.head_dim))
        k = self.k_norm(self.k_proj(x).view(B, T, self.n_kv, self.head_dim))
        v = self.v_proj(x).view(B, T, self.n_kv, self.head_dim)
        cos, sin = self.rope(T, x.device, pos)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            cache.k[:, :, pos:pos + T] = k.transpose(1, 2).to(cache.k.dtype)
            cache.v[:, :, pos:pos + T] = v.transpose(1, 2).to(cache.v.dtype)
            if pos > 0:
                return self._attend_cache(q, cache.k[:, :, :pos + T], cache.v[:, :, :pos + T], pos)
        if self.n_kv != self.n_q:
            rep = self.n_q // self.n_kv
            k = k.repeat_interleave(rep, dim=2)
            v = v.repeat_interleave(rep, dim=2)
        o = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), is_causal=True)
        return o.transpose(1, 2).reshape(B, T, self.n_q * self.head_dim)

    def _attend_cache(self, q, K, V, pos: int):
        """q [B,T,n_q,D] at positions pos..pos+T-1; K,V [B,n_kv,pos+T,D] (cache incl. the new positions)."""
        B, T = q.shape[:2]
        rep, D = self.n_q // self.n_kv, self.head_dim
        if T == 1:   # decode: grouped-query attention without copying the cache (query head i uses KV head i//rep):
            # the rep query heads of one KV head act as rep "query positions" that all see the whole cache
            qg = q.reshape(B, self.n_kv, rep, D).to(K.dtype)
            o = F.scaled_dot_product_attention(qg, K, V)                    # [B,n_kv,rep,D]
            return o.reshape(B, 1, self.n_q * D).to(q.dtype)
        # chunk with history (chunked prefill): explicit mask, SDPA's is_causal is aligned top-left
        L = K.shape[2]
        mask = torch.arange(L, device=q.device)[None, :] <= (pos + torch.arange(T, device=q.device))[:, None]
        if rep > 1:
            K, V = K.repeat_interleave(rep, dim=1), V.repeat_interleave(rep, dim=1)
        o = F.scaled_dot_product_attention(q.transpose(1, 2), K.to(q.dtype), V.to(q.dtype), attn_mask=mask)
        return o.transpose(1, 2).reshape(B, T, self.n_q * D)


# ---------------------------------------------------------------------------
# Blocks
# ---------------------------------------------------------------------------
class HybridMixer(nn.Module):
    def __init__(self, cfg: ModelConfig, n_gdn: int, n_attn: int, n_kv: int):
        super().__init__()
        self.gdn = (GatedDeltaNetHeads(cfg.d_model, n_gdn, cfg.head_dim, cfg.conv_size, cfg.gdn_backend, cfg.gdn_chunk)
                    if n_gdn > 0 else None)
        self.attn = (AttentionHeads(cfg.d_model, n_attn, n_kv, cfg.head_dim, cfg.rope_theta, cfg.qk_norm)
                     if n_attn > 0 else None)
        self.o_proj = nn.Linear((n_gdn + n_attn) * cfg.head_dim, cfg.d_model, bias=False)
        self.o_proj._is_residual_out = True

    def forward(self, x, cache=None, pos: int = 0):
        outs = []
        if self.gdn is not None:
            outs.append(self.gdn(x, cache))
        if self.attn is not None:
            outs.append(self.attn(x, cache, pos))
        return self.o_proj(torch.cat(outs, dim=-1) if len(outs) > 1 else outs[0])


class SwiGLU(nn.Module):
    def __init__(self, d: int, hidden: int):
        super().__init__()
        self.gate = nn.Linear(d, hidden, bias=False)
        self.up = nn.Linear(d, hidden, bias=False)
        self.down = nn.Linear(hidden, d, bias=False)
        self.down._is_residual_out = True

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, cfg: ModelConfig, plan: Dict[str, int]):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.mixer = HybridMixer(cfg, plan["gdn"], plan["attn"], plan["kv"])
        self.norm2 = RMSNorm(cfg.d_model)
        self.mlp = SwiGLU(cfg.d_model, cfg.mlp_hidden)

    def forward(self, x, cache=None, pos: int = 0):
        x = x + self.mixer(self.norm1(x), cache, pos)
        return x + self.mlp(self.norm2(x))


# ---------------------------------------------------------------------------
# ICE modules
# ---------------------------------------------------------------------------
class ICEInput(nn.Module):
    def __init__(self, d: int, n_orders: int, trust: bool = True):
        super().__init__()
        self.W = nn.Parameter(torch.zeros(n_orders, d, d))      # zero init: ICE starts silent
        self.L = nn.Embedding(N_LEN_BUCKETS, d)
        self.D = nn.Embedding(N_DIST_BUCKETS, d)
        nn.init.zeros_(self.L.weight)
        nn.init.zeros_(self.D.weight)
        self.n_orders, self.trust = n_orders, trust
        if trust:   # per-order "trust" embeddings (agreement of occurrences, online reliability)
            self.A = nn.Embedding(n_orders * N_AGREE_BUCKETS, d)
            self.R = nn.Embedding(n_orders * N_REL_BUCKETS, d)
            nn.init.zeros_(self.A.weight)
            nn.init.zeros_(self.R.weight)
        self.norm_h = RMSNorm(d)
        self.norm_f = RMSNorm(d)
        self.d = d

    def forward(self, h, tok_emb: nn.Embedding, succ, len_b, dist_b, keep, agree=None, rel=None):
        # h [B,T,d]; succ/len_b/dist_b/agree/rel [B,N,T]; keep [B]
        m = (succ >= 0).to(h.dtype)                                    # [B,N,T]
        e = tok_emb(succ.clamp(min=0))                                 # [B,N,T,d]
        f = torch.einsum("bntd,ned->bnte", e, self.W.to(e.dtype))      # W_n E[s_n]
        f = f + self.L(len_b) + self.D(dist_b)
        if self.trust:
            order = torch.arange(self.n_orders, device=succ.device)[None, :, None]
            f = f + self.A(order * N_AGREE_BUCKETS + agree) + self.R(order * N_REL_BUCKETS + rel)
        hn = self.norm_h(h).unsqueeze(1)                               # [B,1,T,d]
        g = torch.sigmoid((hn * self.norm_f(f)).float().sum(-1) / math.sqrt(self.d))   # [B,N,T]
        add = ((m * g.to(h.dtype)).unsqueeze(-1) * f).sum(1)           # [B,T,d]
        return h + keep.to(h.dtype)[:, None, None] * add


class ICEOutput(nn.Module):
    def __init__(self, d: int, trust: bool = True):
        super().__init__()
        self.norm = RMSNorm(d)
        self.w = nn.Linear(d, 1, bias=False)
        self.b = nn.Embedding(N_LEN_BUCKETS, 1)
        nn.init.zeros_(self.w.weight)
        nn.init.constant_(self.b.weight, -2.0)     # lambda ~ 0.13 at init
        self.trust = trust
        if trust:
            self.a = nn.Embedding(N_AGREE_BUCKETS, 1)
            self.r = nn.Embedding(N_REL_BUCKETS, 1)
            nn.init.zeros_(self.a.weight)
            nn.init.zeros_(self.r.weight)

    def forward(self, logits, h_last, s_star, len_star, keep, agree_star=None, rel_star=None):
        pre = self.w(self.norm(h_last)).float().squeeze(-1) + self.b(len_star).float().squeeze(-1)
        if self.trust:
            pre = pre + self.a(agree_star).float().squeeze(-1) + self.r(rel_star).float().squeeze(-1)
        lam = F.softplus(pre)
        lam = lam * (s_star >= 0).float() * keep.float()[:, None]      # [B,T]
        return logits.scatter_add(-1, s_star.clamp(min=0).unsqueeze(-1), lam.unsqueeze(-1).to(logits.dtype))


# ---------------------------------------------------------------------------
# Inference cache
# ---------------------------------------------------------------------------
class LayerCache:
    """Decode state of one block: k, v [B, n_kv, max_len, D] (attention heads); conv [B, conv-1, 3*inner] and
    S [B, H, D, D] fp32 (GDN heads). Unused fields stay None."""

    def __init__(self):
        self.k = self.v = self.conv = self.S = None

    def tensors(self):
        return {n: t for n in ("k", "v", "conv", "S") if (t := getattr(self, n)) is not None}


class InferenceCache:
    def __init__(self, layers: List[LayerCache], max_len: int):
        self.layers, self.max_len, self.pos = layers, max_len, 0

    def nbytes(self) -> Dict[str, int]:
        """Bytes per kind: kv (grows with max_len), gdn_state + conv (fixed size)."""
        out = dict(kv=0, gdn_state=0, conv=0)
        for lc in self.layers:
            for n, t in lc.tensors().items():
                out["kv" if n in ("k", "v") else ("gdn_state" if n == "S" else "conv")] += t.numel() * t.element_size()
        return out


# ---------------------------------------------------------------------------
# Language model
# ---------------------------------------------------------------------------
class ICELM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.plan = layer_plan(cfg)
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg, p) for p in self.plan])
        self.norm = RMSNorm(cfg.d_model)
        self.lm_head = None if cfg.tie_embeddings else nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        n_orders = len(cfg.ice_orders)
        self.ice_in = ICEInput(cfg.d_model, n_orders, cfg.ice_trust) if (cfg.ice and cfg.ice_input) else None
        self.ice_out = ICEOutput(cfg.d_model, cfg.ice_trust) if (cfg.ice and cfg.ice_output) else None
        self._init_weights()

    def _init_weights(self):
        std = self.cfg.init_std
        for name, mod in self.named_modules():
            if name.startswith("ice_"):
                continue
            if isinstance(mod, nn.Linear):
                s = std / math.sqrt(2 * self.cfg.n_layers) if getattr(mod, "_is_residual_out", False) else std
                nn.init.normal_(mod.weight, 0.0, s)
            elif isinstance(mod, nn.Embedding):
                nn.init.normal_(mod.weight, 0.0, std)

    # -- utilities -------------------------------------------------------------
    def prepare_backend(self, device, dtype):
        """Run the GDN kernel self-test once (outside of autograd) for this device/dtype."""
        if any(p["gdn"] > 0 for p in self.plan):
            set_torch_kernel_compile(self.cfg.gdn_compile)
            resolve_backend(self.cfg.gdn_backend, torch.device(device), dtype, self.cfg.head_dim)

    def num_params(self, non_embedding: bool = False) -> int:
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.embed.weight.numel()
        return n

    def flops_per_token(self, seq_len: int) -> float:
        """Approximate training FLOPs per token (forward + backward = 3x forward): 6 x (non-embedding params + the
        output projection) + causal attention (6 * seq_len * attention width per layer) + GDN state updates
        (~18 * head_dim^2 per GDN head). The comparison between arms is at equal tokens; report this next to it."""
        n = self.num_params(non_embedding=True) + self.embed.weight.numel()      # tied output projection
        attn = sum(6 * seq_len * p["attn"] * self.cfg.head_dim for p in self.plan)
        gdn = sum(18 * p["gdn"] * self.cfg.head_dim ** 2 for p in self.plan)
        return float(6 * n + attn + gdn)

    def ice_params(self) -> int:
        return sum(p.numel() for m in (self.ice_in, self.ice_out) if m is not None for p in m.parameters())

    # -- inference cache -------------------------------------------------------
    def allocate_cache(self, batch: int, max_len: int, device, dtype=torch.float16) -> "InferenceCache":
        """Preallocated decode state: KV cache for attention heads, conv + recurrent state for GDN heads."""
        layers = []
        for blk, p in zip(self.blocks, self.plan):
            lc = LayerCache()
            if p["attn"] > 0:
                shape = (batch, p["kv"], max_len, self.cfg.head_dim)
                lc.k = torch.zeros(shape, device=device, dtype=dtype)
                lc.v = torch.zeros(shape, device=device, dtype=dtype)
            if p["gdn"] > 0:
                inner = p["gdn"] * self.cfg.head_dim
                lc.conv = torch.zeros(batch, self.cfg.conv_size - 1, 3 * inner, device=device, dtype=dtype)
                lc.S = torch.zeros(batch, p["gdn"], self.cfg.head_dim, self.cfg.head_dim, device=device,
                                   dtype=torch.float32)
            layers.append(lc)
        return InferenceCache(layers, max_len)

    @torch.no_grad()
    def forward_cached(self, input_ids, cache: "InferenceCache", ice: Optional[Dict[str, torch.Tensor]] = None,
                       last_only: bool = True):
        """Process input_ids [B,T] at positions cache.pos .. cache.pos+T-1 (T > 1: prefill / chunked prefill,
        T == 1: decode) and advance the cache. ICE features are those of these T positions (ice_lm.inference
        .ICEIndex computes them incrementally). Returns logits [B,1,V] (last_only) or [B,T,V]."""
        B, T = input_ids.shape
        pos = cache.pos
        if pos + T > cache.max_len:
            raise ValueError(f"cache holds {cache.max_len} positions, need {pos + T}")
        use_ice = self.ice_in is not None or self.ice_out is not None
        if use_ice and ice is None:
            raise ValueError("this model uses ICE: pass the ICE features of the new positions")
        keep = torch.ones(B, device=input_ids.device)
        h = self.embed(input_ids)
        for i, blk in enumerate(self.blocks):
            h = blk(h, cache.layers[i], pos)
            if self.ice_in is not None and i + 1 == self.cfg.ice_inject_after:
                h = self.ice_in(h, self.embed, ice["ice_succ"], ice["ice_len"], ice["ice_dist"], keep,
                                ice.get("ice_agree"), ice.get("ice_rel"))
        h_last = self.norm(h)
        star = tuple(ice.get(k) for k in OUT_KEYS) if self.ice_out is not None else (None,) * len(OUT_KEYS)
        if last_only:
            h_last = h_last[:, -1:]
            star = tuple(None if f is None else f[:, -1:] for f in star)
        cache.pos = pos + T
        return self._logits(h_last, keep, *star)

    # -- forward ---------------------------------------------------------------
    def forward(self, input_ids, labels=None, ice: Optional[Dict[str, torch.Tensor]] = None,
                ice_keep: Optional[torch.Tensor] = None, reduction: str = "mean"):
        B, T = input_ids.shape
        use_ice = self.ice_in is not None or self.ice_out is not None
        if use_ice and ice is None:
            raise ValueError("this model uses ICE: pass the ICE feature dict (see ice_lm.data)")
        if use_ice and ice_keep is None:
            ice_keep = torch.ones(B, device=input_ids.device)
        h = self.embed(input_ids)
        for i, blk in enumerate(self.blocks):
            if self.cfg.grad_checkpointing and self.training:
                h = checkpoint(blk, h, use_reentrant=False)
            else:
                h = blk(h)
            if self.ice_in is not None and i + 1 == self.cfg.ice_inject_after:
                h = self.ice_in(h, self.embed, ice["ice_succ"], ice["ice_len"], ice["ice_dist"], ice_keep,
                                ice.get("ice_agree"), ice.get("ice_rel"))
        h_last = self.norm(h)
        # output-side ICE features, each [B,T] (or None): s*, len*, agree*, rel*
        star = tuple(ice.get(k) for k in OUT_KEYS) if self.ice_out is not None else (None,) * len(OUT_KEYS)
        if labels is not None and reduction == "mean" and torch.is_grad_enabled():
            # training: never hold the full fp32 [B,T,V] logits (2 x 500 MB at T=4096, V=32K); compute the head
            # and the loss in chunks and recompute them in backward
            total = h_last.new_zeros((), dtype=torch.float32)
            for t0 in range(0, T, self.LOSS_CHUNK):
                sl = slice(t0, t0 + self.LOSS_CHUNK)
                total = total + checkpoint(self._loss_sum, h_last[:, sl], labels[:, sl], ice_keep,
                                           *(None if f is None else f[:, sl] for f in star), use_reentrant=False)
            return None, total / (labels != -100).sum().clamp_min(1)
        logits = self._logits(h_last, ice_keep, *star)
        if labels is None:
            return logits
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.reshape(-1), reduction=reduction)
        if reduction == "none":
            loss = loss.view(B, T)
        return logits, loss

    LOSS_CHUNK = 1024

    def _logits(self, h_last, keep, s_star=None, len_star=None, agree_star=None, rel_star=None):
        w = self.embed.weight if self.lm_head is None else self.lm_head.weight
        logits = F.linear(h_last, w).float()
        if self.ice_out is not None:
            logits = self.ice_out(logits, h_last, s_star, len_star, keep, agree_star, rel_star)
        return logits

    def _loss_sum(self, h_last, labels, keep, *star):
        logits = self._logits(h_last, keep, *star)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), reduction="sum")


In [ ]:
%%writefile ice_lm/ice_lm/data.py
"""Data: tokenize a prose + code mixture into flat uint16/uint32 token files, then serve packed
windows with ICE features computed on the fly (CPU, in DataLoader workers).

Files written by `prepare_data` into data_dir:
    train.bin, val_prose.bin, val_code.bin, meta.json
"""
from __future__ import annotations

import json
import math
import os
import time
from typing import Dict, Iterator, Optional

import numpy as np
import torch
from torch.utils.data import IterableDataset

from .ice_features import compute_ice_features, segments_from_eos

PROSE_DEFAULT = dict(path="HuggingFaceFW/fineweb-edu", name="sample-10BT", field="text")
CODE_DEFAULT = dict(path="codeparrot/codeparrot-clean", name=None, field="content")
TOKENIZER_DEFAULT = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   # Llama tokenizer, 32K vocab, ungated


# ---------------------------------------------------------------------------
# Preparation
# ---------------------------------------------------------------------------
class _Writer:
    def __init__(self, path: str, dtype):
        self.f = open(path, "wb")
        self.dtype = dtype
        self.n = 0

    def write(self, ids):
        a = np.asarray(ids, dtype=self.dtype)
        self.f.write(a.tobytes())
        self.n += len(a)

    def close(self):
        self.f.close()


def _doc_stream(spec: dict, seed: int) -> Iterator[str]:
    from datasets import load_dataset
    kw = dict(split="train", streaming=True)
    ds = load_dataset(spec["path"], spec.get("name"), **kw)
    ds = ds.shuffle(seed=seed, buffer_size=10_000)
    for ex in ds:
        txt = ex.get(spec["field"])
        if txt:
            yield txt


def prepare_data(out_dir: str, train_tokens: int, val_tokens: int = 2_000_000, code_frac: float = 0.3,
                 tokenizer_name: str = TOKENIZER_DEFAULT, prose: Optional[dict] = None,
                 code: Optional[dict] = None, seed: int = 0, batch_docs: int = 256,
                 max_doc_chars: int = 200_000) -> dict:
    """Stream + tokenize. Validation sets (prose and code separately) are taken first."""
    from transformers import AutoTokenizer
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
    os.makedirs(out_dir, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(tokenizer_name)
    eos = tok.eos_token_id
    vocab = len(tok)
    dtype = np.uint16 if vocab <= 65535 else np.uint32
    prose = prose or PROSE_DEFAULT
    code = code or CODE_DEFAULT
    streams = {"prose": _doc_stream(prose, seed), "code": _doc_stream(code, seed)}

    def batches(name):
        buf = []
        for txt in streams[name]:
            buf.append(txt[:max_doc_chars])
            if len(buf) == batch_docs:
                yield buf
                buf = []
        if buf:
            yield buf

    gens = {k: batches(k) for k in streams}

    def tokenize(texts):
        out = tok(texts, add_special_tokens=False)["input_ids"]
        return [ids + [eos] for ids in out]

    t0 = time.time()
    # validation sets
    for name in ("prose", "code"):
        w = _Writer(os.path.join(out_dir, f"val_{name}.bin"), dtype)
        while w.n < val_tokens:
            docs = next(gens[name], None)
            if docs is None:
                raise RuntimeError(f"{name} stream ended before the validation set was complete")
            for ids in tokenize(docs):
                w.write(ids)
        w.close()
        print(f"[data] val_{name}: {w.n:,} tokens")
    # training mixture, balanced by token counts
    w = _Writer(os.path.join(out_dir, "train.bin"), dtype)
    counts = {"prose": 0, "code": 0}
    exhausted = set()
    last = 0
    while w.n < train_tokens:
        name = "code" if counts["code"] < code_frac * max(1, w.n) else "prose"
        if name in exhausted:
            name = "prose" if name == "code" else "code"
        raw = next(gens[name], None)
        if raw is None:
            exhausted.add(name)
            print(f"[data] {name} stream exhausted; continuing with the other one (mix will drift)")
            if len(exhausted) == 2:
                print(f"[data] both streams exhausted at {w.n:,} tokens")
                break
            continue
        docs = tokenize(raw)
        for ids in docs:
            w.write(ids)
            counts[name] += len(ids)
        if w.n - last > 50_000_000:
            last = w.n
            el = time.time() - t0
            print(f"[data] train {w.n/1e6:,.0f}M / {train_tokens/1e6:,.0f}M tokens "
                  f"({w.n/el/1e6:.2f}M tok/s, code {counts['code']/max(1,w.n):.0%})", flush=True)
    w.close()
    meta = dict(tokenizer=tokenizer_name, vocab_size=vocab, eos_id=eos, dtype=np.dtype(dtype).name,
                train_tokens=w.n, code_frac_actual=counts["code"] / max(1, w.n), prose=prose, code=code,
                fingerprint=data_fingerprint(out_dir, dtype))
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(f"[data] done in {time.time()-t0:.0f}s: {meta}")
    return meta


def make_synthetic_data(out_dir: str, train_tokens: int = 3_000_000, val_tokens: int = 200_000,
                        vocab_size: int = 512, seed: int = 0) -> dict:
    """Offline copy-heavy toy corpus (for smoke tests): documents that re-use earlier spans.
    'code' docs repeat spans often; 'prose' docs rarely. ICE should help on this data."""
    rng = np.random.default_rng(seed)
    os.makedirs(out_dir, exist_ok=True)
    eos = vocab_size - 1

    def doc(copy_prob):
        n = int(rng.integers(200, 1200))
        out = []
        while len(out) < n:
            if len(out) > 20 and rng.random() < copy_prob:
                L = int(rng.integers(8, 40))
                s = int(rng.integers(0, max(1, len(out) - L)))
                out.extend(out[s:s + L])
            else:
                # locally predictable "words": small Markov-ish chunks
                out.extend(rng.integers(1, vocab_size - 1, size=int(rng.integers(3, 10))).tolist())
        return out[:n] + [eos]

    def write(path, n_tok, copy_prob_fn):
        w = _Writer(path, np.uint16)
        while w.n < n_tok:
            w.write(doc(copy_prob_fn()))
        w.close()
        return w.n

    write(os.path.join(out_dir, "val_prose.bin"), val_tokens, lambda: 0.05)
    write(os.path.join(out_dir, "val_code.bin"), val_tokens, lambda: 0.5)
    n = write(os.path.join(out_dir, "train.bin"), train_tokens, lambda: 0.5 if rng.random() < 0.3 else 0.05)
    meta = dict(tokenizer=None, vocab_size=vocab_size, eos_id=eos, dtype="uint16", train_tokens=n, synthetic=True)
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    return meta


def data_fingerprint(data_dir: str, dtype) -> str:
    """Cheap content fingerprint of the token files: runs (sessions) that re-downloaded the data can check that
    they trained on exactly the same tokens (streaming datasets are not pinned to a revision)."""
    parts = []
    for split in ("train", "val_prose", "val_code"):
        x = np.memmap(os.path.join(data_dir, f"{split}.bin"), dtype=dtype, mode="r")
        n = min(len(x), 1 << 20)
        idx = np.arange(n, dtype=np.int64) * max(0, len(x) - 1) // max(1, n - 1)
        parts.append(f"{len(x)}:{int(np.asarray(x[idx], dtype=np.int64).sum() % (1 << 61))}")
    return "|".join(parts)


def load_meta(data_dir: str) -> dict:
    with open(os.path.join(data_dir, "meta.json")) as f:
        return json.load(f)


def open_tokens(data_dir: str, split: str) -> np.memmap:
    meta = load_meta(data_dir)
    return np.memmap(os.path.join(data_dir, f"{split}.bin"), dtype=np.dtype(meta["dtype"]), mode="r")


# ---------------------------------------------------------------------------
# Datasets
# ---------------------------------------------------------------------------
ICE_KEYS = ("ice_succ", "ice_len", "ice_dist", "ice_agree", "ice_rel",
            "ice_s_star", "ice_len_star", "ice_agree_star", "ice_rel_star")
EVAL_KEYS = ("ice_match_len", "ice_dist_star", "lookup_s_star")


def make_example(window: np.ndarray, eos_id: Optional[int], orders, with_ice: bool) -> Dict[str, torch.Tensor]:
    """window: seq_len + 1 tokens -> input_ids, labels (+ ICE features for the inputs)."""
    w = window.astype(np.int64)
    x, y = w[:-1], w[1:]
    ex = dict(input_ids=torch.from_numpy(x), labels=torch.from_numpy(y))
    feats = compute_ice_features(x, segments_from_eos(x, eos_id), orders)
    feats["lookup_s_star"] = feats["ice_s_star"]
    # lookup statistics are always returned (eval splits: copyable / lookup right / lookup misleading)
    for k in EVAL_KEYS:
        ex[k] = torch.from_numpy(feats[k])
    if with_ice:
        for k in ICE_KEYS:
            ex[k] = torch.from_numpy(feats[k])
    return ex


class TrainWindows(IterableDataset):
    """Infinite stream of training windows (each non-overlapping window once per pass over the data).

    Samples are indexed globally: micro-batch m (counted over the whole run) of rank r holds the sample indices
    (m * world + r) * micro_batch + j, j < micro_batch. DataLoader worker w of nw produces this rank's local
    micro-batches w, w + nw, ... in order. Optimizer step s therefore always uses the sample indices
    [s*S, (s+1)*S), S = micro_batch * grad_accum * world: the data per step depends only on the seed and the global
    batch, not on micro-batch, GPU count or num_workers, and a resume at start_index = samples consumed continues
    exactly where the run stopped."""

    def __init__(self, data_dir: str, seq_len: int, orders, with_ice: bool, ice_dropout: float,
                 seed: int, rank: int = 0, world: int = 1, start_index: int = 0, micro_batch: int = 1):
        self.data_dir, self.seq_len, self.orders = data_dir, seq_len, tuple(orders)
        self.with_ice, self.ice_dropout = with_ice, ice_dropout
        self.seed, self.rank, self.world, self.start_index = seed, rank, world, start_index
        self.micro_batch = micro_batch
        self.eos = load_meta(data_dir)["eos_id"]

    def sample(self, data, i: int, cache: dict):
        """Sample index i -> window perm_e[i mod n_win] of pass e = i // n_win: every window once per pass, a new
        order (and shifted window boundaries) in later passes. The same index gives the same window in every arm."""
        L = self.seq_len
        n_win = max(1, (len(data) - 1) // L - 1)            # non-overlapping windows (one spare for the shift)
        epoch, k = divmod(i, n_win)
        if cache.get("epoch") != epoch:
            prng = np.random.default_rng([self.seed, 7919, epoch])
            cache.update(epoch=epoch, perm=prng.permutation(n_win),
                         shift=0 if epoch == 0 else int(prng.integers(0, L)))
        off = int(cache["perm"][k]) * L + cache["shift"]
        ex = make_example(np.asarray(data[off: off + L + 1]), self.eos, self.orders, self.with_ice)
        ex["ice_keep"] = torch.tensor(float(np.random.default_rng([self.seed, i]).random() >= self.ice_dropout))
        return ex

    def __iter__(self):
        data = open_tokens(self.data_dir, "train")
        info = torch.utils.data.get_worker_info()
        wid, nw = (info.id, info.num_workers) if info else (0, 1)
        mb, cache = self.micro_batch, {}
        m = self.start_index // (mb * self.world) + wid       # this worker's first global micro-batch
        while True:
            base = (m * self.world + self.rank) * mb
            for j in range(mb):
                yield self.sample(data, base + j, cache)
            m += nw


def val_batches(data_dir: str, split: str, seq_len: int, batch_size: int, n_batches: Optional[int],
                orders, with_ice: bool):
    """Sequential non-overlapping windows from a validation file."""
    data = open_tokens(data_dir, split)
    eos = load_meta(data_dir)["eos_id"]
    n_windows = (len(data) - 1) // seq_len
    if n_batches is not None:
        n_windows = min(n_windows, n_batches * batch_size)
    batch = []
    for j in range(n_windows):
        w = np.asarray(data[j * seq_len: j * seq_len + seq_len + 1])
        ex = make_example(w, eos, orders, with_ice)
        ex["ice_keep"] = torch.tensor(1.0)
        batch.append(ex)
        if len(batch) == batch_size:
            yield collate(batch)
            batch = []
    if batch:
        yield collate(batch)


def collate(examples):
    out = {}
    for k in examples[0]:
        out[k] = torch.stack([e[k] for e in examples])
    return out




def split_batch(batch: Dict[str, torch.Tensor], device, with_stats: bool = False):
    """-> input_ids, labels, ice dict (or None), ice_keep, match_len (all on device)
    (+ dict of lookup statistics for the eval splits when with_stats)."""
    b = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
    ice = {k: b[k] for k in ICE_KEYS} if "ice_succ" in b else None
    out = (b["input_ids"], b["labels"], ice, b.get("ice_keep"), b.get("ice_match_len"))
    return out + ({k: b[k] for k in EVAL_KEYS if k in b},) if with_stats else out


In [ ]:
%%writefile ice_lm/ice_lm/evals.py
"""Evaluations from the proposal (Section 2.2, Metrics):
  (i)  held-out loss split by prose / code and by "copyable" tokens (longest match >= 4) vs the rest,
       and by what the lookup says about the next token: right / misleading / no match, with the
       "follow rate" P(argmax == s* | s* is wrong) -- how often the model is fooled by a wrong lookup
  (ii) exact-copy tasks: repeat-a-string, phonebook, MQAR (also beyond the training length)
  (iv) needle retrieval with TEMPLATED vs PARAPHRASED questions
All task scores are teacher-forced: answer-token accuracy and exact match (all answer tokens right).
"""
from __future__ import annotations

import math
from contextlib import nullcontext
from typing import Dict, List, Optional

import numpy as np
import torch

from .data import ICE_KEYS, load_meta, open_tokens, split_batch, val_batches
from .ice_features import compute_ice_features


def _autocast(device, dtype):
    if device.type == "cuda" and dtype in (torch.float16, torch.bfloat16):
        return torch.autocast("cuda", dtype=dtype)
    return nullcontext()


def uses_ice(model) -> bool:
    return model.ice_in is not None or model.ice_out is not None


@torch.no_grad()
def eval_loss_splits(model, data_dir: str, seq_len: int, batch_size: int, n_batches: Optional[int],
                     device, amp_dtype, copy_min_len: int = 4, far: int = 512,
                     ice_keep: float = 1.0) -> Dict[str, float]:
    """Loss (and accuracy) per token group. Groups are defined by the exact lookup alone, so they are the same
    tokens for every arm (with or without ICE). ice_keep=0 evaluates an ICE model with ICE switched off."""
    model.eval()
    with_ice = uses_ice(model)
    res = {}
    for split, name in (("val_prose", "prose"), ("val_code", "code")):
        groups = ("all", "copyable", "other", "copy_far", "lk_right", "lk_wrong", "lk_none")
        tot = {g: [0.0, 0, 0] for g in groups}          # loss sum, count, correct
        follow = 0
        for batch in val_batches(data_dir, split, seq_len, batch_size, n_batches, model.cfg.ice_orders, with_ice):
            x, y, ice, keep, mlen, st = split_batch(batch, device, with_stats=True)
            if keep is not None:
                keep = torch.full_like(keep, float(ice_keep))
            with _autocast(device, amp_dtype):
                logits, loss = model(x, y, ice=ice, ice_keep=keep, reduction="none")
            loss = loss.float()
            pred = logits.argmax(-1)
            s_star = st["lookup_s_star"]
            cp = mlen >= copy_min_len
            masks = dict(all=torch.ones_like(cp), copyable=cp, other=~cp, copy_far=cp & (st["ice_dist_star"] > far),
                         lk_right=s_star == y, lk_wrong=(s_star >= 0) & (s_star != y), lk_none=s_star < 0)
            for key, mask in masks.items():
                tot[key][0] += loss[mask].sum().item()
                tot[key][1] += int(mask.sum().item())
                tot[key][2] += int((pred == y)[mask].sum().item())
            follow += int((pred == s_star)[masks["lk_wrong"]].sum().item())
        for key, (s, n, c) in tot.items():
            res[f"{name}/{key}_loss"] = s / n if n else float("nan")      # nan: no such tokens (e.g. copy_far)
            res[f"{name}/{key}_acc"] = c / n if n else float("nan")
            res[f"{name}/{key}_frac"] = n / max(1, tot["all"][1])
        # how often the model's top prediction is the (wrong) lookup token
        res[f"{name}/follow_wrong_lookup"] = follow / tot["lk_wrong"][1] if tot["lk_wrong"][1] else float("nan")
    model.train()
    return res


# ---------------------------------------------------------------------------
# Synthetic tasks
# ---------------------------------------------------------------------------
@torch.no_grad()
def score_answers(model, seqs: List[List[int]], answer_spans: List, device, amp_dtype, ice_keep: float = 1.0,
                  details: bool = False, decide: Optional[List[int]] = None):
    """seqs[i] = full token list; answer_spans[i] = (start, end) or a list of (start, end) spans of answer
    tokens in seqs[i]. Returns (token accuracy, exact-match rate over spans, mean answer log-prob), or with
    details=True a dict that also splits answer-token accuracy by what the exact lookup suggested for that token
    (right / wrong / none) and gives the rate of following a wrong lookup (argmax == the wrong s*).
    decide[i] (optional): position of the one token that decides item i (e.g. the first digit of a value: later
    digits can be copied once it is given). Adds dec_* metrics split by the lookup status of that token and the
    per-item 0/1 list items_dec (for paired comparisons between arms, which score identical items)."""
    model.eval()
    with_ice = uses_ice(model)
    acc_n = acc_d = em = n_spans = 0
    lp_sum = 0.0
    by = {s: [0, 0] for s in ("right", "wrong", "none")}   # answer tokens, correct
    dby = {s: [0, 0] for s in ("right", "wrong", "none")}  # deciding tokens, correct
    follow = dfollow = 0
    items_dec: List[int] = []
    for i_item, (seq, spans) in enumerate(zip(seqs, answer_spans)):
        spans = [spans] if isinstance(spans, tuple) else list(spans)
        x = torch.tensor(seq[:-1], device=device)[None]
        f = compute_ice_features(np.array(seq[:-1]), None, model.cfg.ice_orders)
        ice = {k: torch.from_numpy(f[k]).to(device)[None] for k in ICE_KEYS} if with_ice else None
        keep = torch.full((1,), float(ice_keep), device=device)
        with _autocast(device, amp_dtype):
            logits = model(x, ice=ice, ice_keep=keep)
        logits = logits[0].float()
        for a, b in spans:
            tgt = torch.tensor(seq[a:b], device=device)
            rows = logits[a - 1: b - 1]                     # position t predicts token t+1
            pred = rows.argmax(-1)
            correct = pred == tgt
            acc_n += int(correct.sum())
            acc_d += len(tgt)
            em += int(correct.all())
            n_spans += 1
            lp_sum += torch.log_softmax(rows, -1).gather(-1, tgt[:, None]).mean().item()
            for i, t in enumerate(range(a, b)):
                s0 = int(f["ice_s_star"][t - 1])
                status = "none" if s0 < 0 else ("right" if s0 == seq[t] else "wrong")
                by[status][0] += 1
                by[status][1] += int(correct[i])
                follow += int(status == "wrong" and int(pred[i]) == s0)
        if decide is not None:
            t = decide[i_item]
            s0 = int(f["ice_s_star"][t - 1])
            status = "none" if s0 < 0 else ("right" if s0 == seq[t] else "wrong")
            p_t = int(logits[t - 1].argmax())
            ok = int(p_t == seq[t])
            dby[status][0] += 1
            dby[status][1] += ok
            dfollow += int(status == "wrong" and p_t == s0)
            items_dec.append(ok)
    model.train()
    acc, em_rate, lp = acc_n / max(1, acc_d), em / max(1, n_spans), lp_sum / max(1, n_spans)
    if not details:
        return acc, em_rate, lp
    out = dict(acc=acc, em=em_rate, logprob=lp, follow_wrong_lookup=follow / max(1, by["wrong"][0]))
    for st, (n_, c_) in by.items():
        out[f"lookup_{st}_frac"] = n_ / max(1, acc_d)
        out[f"acc_lookup_{st}"] = c_ / n_ if n_ else float("nan")
    if decide is not None:
        n_dec = max(1, len(items_dec))
        out["dec_acc"] = sum(items_dec) / n_dec
        out["items_dec"] = items_dec
        out["dec_follow_wrong_lookup"] = dfollow / dby["wrong"][0] if dby["wrong"][0] else float("nan")
        for st, (n_, c_) in dby.items():
            out[f"dec_lookup_{st}_frac"] = n_ / n_dec
            out[f"dec_acc_lookup_{st}"] = c_ / n_ if n_ else float("nan")
    return out


def task_repeat(rng, vocab_lo, vocab_hi, length, sep, prefix_len=16):
    s = rng.integers(vocab_lo, vocab_hi, size=length).tolist()
    pre = rng.integers(vocab_lo, vocab_hi, size=prefix_len).tolist()
    seq = pre + s + sep + s
    a = len(pre) + len(s) + len(sep) + 1         # skip the first copied token (not predictable)
    return seq, (a, len(seq))


def task_mqar(rng, vocab_lo, vocab_hi, n_pairs, n_queries):
    keys = rng.choice(np.arange(vocab_lo, vocab_hi), size=n_pairs, replace=False).tolist()
    vals = rng.integers(vocab_lo, vocab_hi, size=n_pairs).tolist()
    seq = []
    for k_, v_ in zip(keys, vals):
        seq += [k_, v_]
    q = rng.choice(n_pairs, size=n_queries, replace=False)
    spans = []
    for j in q:
        seq += [keys[j], vals[j]]
        spans.append(len(seq) - 1)
    return seq, spans


_NAMES = ["Alice", "Bruno", "Chen", "Dara", "Elif", "Farid", "Gita", "Hugo", "Ines", "Jonas", "Kenji", "Lena",
          "Mateo", "Nadia", "Omar", "Priya", "Quinn", "Rosa", "Sven", "Tara", "Umar", "Vera", "Wen", "Xavi",
          "Yara", "Zane"]
_SURN = ["Nguyen", "Smith", "Okafor", "Garcia", "Kowalski", "Tanaka", "Haddad", "Silva", "Novak", "Rossi",
         "Ivanova", "Kim", "Murphy", "Sato", "Dubois", "Moreau", "Larsen", "Costa", "Fischer", "Patel"]
_COLORS = ["red", "blue", "green", "yellow", "purple", "orange", "black", "white", "silver", "golden"]
_OBJECTS = ["door", "box", "safe", "gate", "locker", "chest", "drawer", "vault", "cabinet", "case"]


def encode_with_answer(tok, prompt: str, answer: str):
    """Tokenize prompt+answer together; the answer span starts where the prompt's own tokenization stops
    matching (robust to tokenizers that merge a leading space into the answer)."""
    full = tok(prompt + answer, add_special_tokens=False)["input_ids"]
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = 0
    while a < min(len(p), len(full)) and p[a] == full[a]:
        a += 1
    a = min(a, len(p))
    while a < len(full) - 1 and not tok.decode([full[a]]).strip():      # a bare "▁" / space token is free
        a += 1
    return full, (a, len(full))


def build_text_tasks(tok, rng, n: int, context_len: int, filler: np.ndarray, eos_id: int):
    """Returns dict task_name -> (seqs, spans) for phonebook / needle-templated / needle-paraphrased."""
    out = {}
    # phonebook: fill the context with "Name Surname: 7 digits" lines, then ask for one entry
    seqs, spans = [], []
    line_tokens = max(8, len(tok("Alice Nguyen Kowalski: 1234567\n", add_special_tokens=False)["input_ids"]))
    n_lines = max(4, (context_len - 32) // line_tokens)
    for _ in range(n):
        book = {}
        while len(book) < n_lines:
            book[f"{rng.choice(_NAMES)} {rng.choice(_SURN)} {rng.choice(_SURN)}"] = \
                "".join(str(d) for d in rng.integers(0, 10, size=7))
        names = list(book)
        name = names[int(rng.integers(0, len(names)))]
        prompt = "Phone book:\n" + "".join(f"{k}: {v}\n" for k, v in book.items()) + f"\n{name}:"
        ids, span = encode_with_answer(tok, prompt, f" {book[name]}")
        seqs.append(ids)
        spans.append(span)
    out["phonebook"] = (seqs, spans)

    # needles: identical haystacks and needles (same seed), two question styles
    needle_seed = int(rng.integers(1 << 30))
    for style in ("templated", "paraphrased"):
        seqs, spans = [], []
        r2 = np.random.default_rng(needle_seed)
        for _ in range(n):
            color, obj = r2.choice(_COLORS), r2.choice(_OBJECTS)
            num = "".join(str(d) for d in r2.integers(1, 10, size=4))
            needle = tok(f" The secret code for the {color} {obj} is {num}. ", add_special_tokens=False)["input_ids"]
            if style == "templated":
                q = f"\nThe secret code for the {color} {obj} is"
            else:
                q = f"\nQuestion: which number opens the {obj} that is {color}? Answer: the number is"
            q_ids, (qa, qb) = encode_with_answer(tok, q, f" {num}")
            hay_len = max(16, context_len - len(needle) - len(q_ids))
            start = int(r2.integers(0, max(1, len(filler) - hay_len)))
            hay = [t for t in np.asarray(filler[start: start + hay_len]).tolist() if t != eos_id]
            depth = int(r2.integers(0, max(1, len(hay))))
            prefix = hay[:depth] + needle + hay[depth:]
            seqs.append(prefix + q_ids)
            spans.append((len(prefix) + qa, len(prefix) + qb))
        out[f"needle_{style}"] = (seqs, spans)
    return out


def run_synthetic_tasks(model, data_dir: str, device, amp_dtype, context_lens=(1024, 4096, 8192),
                        n: int = 50, seed: int = 0, ice_keep: float = 1.0) -> Dict[str, float]:
    meta = load_meta(data_dir)
    V = model.cfg.vocab_size
    lo, hi = max(10, V // 32), max(20, V - 100)
    rng = np.random.default_rng(seed)
    res = {}
    sep = [lo + 1, lo + 2, lo + 3]
    for L in context_lens:
        # repeat a random string of length ~L/2
        seqs, spans = zip(*[task_repeat(rng, lo, hi, max(8, L // 2 - 16), sep) for _ in range(n)])
        acc, em, lp = score_answers(model, list(seqs), list(spans), device, amp_dtype, ice_keep)
        res[f"repeat@{L}/acc"], res[f"repeat@{L}/em"] = acc, em
        # MQAR: single-token keys/values, n_pairs ~ L/4
        n_pairs = max(16, min((L - 32) // 2, hi - lo - 1))
        mq = [task_mqar(rng, lo, hi, n_pairs=n_pairs, n_queries=16) for _ in range(n)]
        acc, _, _ = score_answers(model, [s_ for s_, _ in mq], [[(p, p + 1) for p in pos] for _, pos in mq],
                                  device, amp_dtype, ice_keep)
        res[f"mqar@{L}/acc"] = acc
        # text tasks need the real tokenizer
        if meta.get("tokenizer"):
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained(meta["tokenizer"])
            filler = open_tokens(data_dir, "val_prose")
            for name, (seqs, spans) in build_text_tasks(tok, rng, n, L, filler, meta["eos_id"]).items():
                acc, em, lp = score_answers(model, seqs, spans, device, amp_dtype, ice_keep)
                res[f"{name}@{L}/acc"], res[f"{name}@{L}/em"] = acc, em
    return res


In [ ]:
%%writefile ice_lm/ice_lm/coderecall.py
"""CodeRecall: one concrete use case for ICE -- recalling identifiers and configuration values in Python code.

Every item is short template code embedded in held-out real code (val_code), scored teacher-forced. Values
are 4-digit numbers and all values inside one item have DIFFERENT leading digits, so the first digit alone says
whether the right value was recalled. Llama-style tokenizers split numbers into digits and, teacher-forced, the
later digits can simply be copied once the first one is given, so the headline metric is accuracy on the
DECIDING token (the first digit; the last digit for `increment`), split by what the exact lookup suggested
for that token (right / wrong / none), plus exact match of the whole value.

  copy_same      value defined once, asked with the SAME syntax           lookup mostly right  (ICE should help)
  copy_diff      value defined once, asked with DIFFERENT wording
                 (settings.x / settings.get("x") / os.environ["X"])        lookup useless (needs attention)
  conflict_g{g}  same key under several objects; the object's name sits g identical kwargs before the key.
                 The target is never the last object, so for large g the lookup (most recent occurrence)
                 points at another object's value                          g=1 right, g>=2 wrong
  override_same  value reassigned later; asked with the same syntax     lookup right   (recency)
  override_diff  value changed later with a different syntax (config.update), asked with the ORIGINAL syntax:
                 the lookup points at the stale value                     lookup wrong   ("revise a value")
  increment      Migration(id=1041), (id=1042), ... next id             last digit: lookup wrong (copying is wrong)
  track          x = 3; x = x + 4; print(x)  # prints 7                   lookup points at an operand (wrong / none)

Statuses are measured per item, not assumed: the split by lookup status is what to read.
Every arm scores identical tokens.

Example (override_diff):
    config["timeout"] = 30
    config["retries"] = 5
    <~D tokens of real code>
    config.update(timeout=60)  # raised after the outage
    <~D tokens of real code>
    # re-apply the current value
    config["timeout"] =  ->  " 60"
"""
from __future__ import annotations

from typing import Dict, List, Optional, Sequence

import numpy as np

from .evals import encode_with_answer, score_answers

KEYS = ["db_port", "cache_ttl", "max_retries", "worker_count", "batch_size", "timeout_ms", "pool_size",
        "queue_limit", "log_every", "shard_count", "page_size", "rate_limit", "buffer_kb", "max_conns",
        "grace_period", "flush_interval", "chunk_bytes", "retry_delay", "lease_secs", "warmup_steps"]
CONSTS = ["TIMEOUT", "RETRIES", "MAX_WORKERS", "PORT", "BATCH", "CACHE_SIZE", "POOL", "LIMIT", "INTERVAL",
          "BACKLOG", "QUOTA", "WINDOW"]
SERVERS = ["dev", "prod", "staging", "qa", "canary", "backup", "edge", "batch", "ml", "api", "web", "db"]
EXTRAS = ['region="eu"', "replicas=3", 'tier="gold"', "healthcheck=True", "weight=10"]
MIGRATIONS = ["add_users", "add_orders", "index_email", "drop_legacy", "add_invoices", "split_names",
              "add_audit", "fix_dates", "add_tags", "merge_carts"]
GAPS = (1, 2, 5)          # with the TinyLlama tokenizer: g=1 lookup right, g>=2 wrong (beyond order 8)
CATEGORIES = ("copy_same", "copy_diff", *(f"conflict_g{g}" for g in GAPS), "override_same", "override_diff",
              "increment", "track")


def _values(rng, n: int) -> List[int]:
    """n <= 9 four-digit values with pairwise different leading digits."""
    lead = rng.choice(np.arange(1, 10), size=n, replace=False)
    return [int(d) * 1000 + int(rng.integers(0, 1000)) for d in lead]




# Each generator returns (parts, answer): parts is a list of strings and None (= a gap of real code); the last
# part is the query, the answer is the expected continuation. `gap` is the TOTAL amount of filler per item (split
# evenly when an item has two gaps), so items stay within the training length (default gaps up to 3072 < 4096):
# beyond it attention must extrapolate while ICE does not, which would confound the comparison.
def gen_copy(rng, n_distract: int, same: bool):
    """Defined once in a dict (or as environment variables); asked with the same or a different wording."""
    keys = list(rng.choice(KEYS, size=min(9, n_distract + 1), replace=False))
    vals = _values(rng, len(keys))
    j = int(rng.integers(len(keys)))
    head = "settings = {\n" + "".join(f'    "{k}": {v},\n' for k, v in zip(keys, vals)) + "}\n"
    if same:
        return [head, None, f'\noverride = {{"{keys[j]}":'], f" {vals[j]}"
    form = int(rng.integers(3))
    if form == 0:
        return [head, None, f"\nassert settings.{keys[j]} =="], f" {vals[j]}"
    if form == 1:
        return [head, None, f'\nlimit = settings.get("{keys[j]}", default='], f"{vals[j]}"
    head = "".join(f'os.environ["{k.upper()}"] = "{v}"\n' for k, v in zip(keys, vals))
    return [head, None, f"\nlimit = int(os.getenv('{keys[j].upper()}', '"], f"{vals[j]}"


def gen_conflict(rng, n_distract: int, gap: int):
    names = list(rng.choice(SERVERS, size=min(9, max(2, n_distract + 1)), replace=False))
    ports = _values(rng, len(names))
    extras = "".join(f"{e}, " for e in EXTRAS[:gap])
    j = int(rng.integers(len(names) - 1))            # never the last (most recent) server
    head = "servers = [\n" + "".join(f'    Server(name="{n}", {extras}port={p}),\n' for n, p in zip(names, ports)) + "]\n"
    query = f'\nprimary = Server(name="{names[j]}", {extras}port='
    return [head, None, query], f"{ports[j]}"


def gen_override(rng, n_distract: int, same: bool):
    if same:
        names = list(rng.choice(CONSTS, size=min(8, n_distract + 1), replace=False))
        vals = _values(rng, len(names) + 1)
        head = "".join(f"{k} = {v}\n" for k, v in zip(names, vals))
        new, j = vals[-1], int(rng.integers(len(names)))          # override a random line, not always the first
        mid = f"\n{names[j]} = {new}  # raised after the outage\n"
        query = f"\n# keep the current value\n{names[j]} ="
    else:
        keys = list(rng.choice(KEYS, size=min(len(KEYS), n_distract + 1), replace=False))[:8]
        vals = _values(rng, len(keys) + 1)
        head = "".join(f'config["{k}"] = {v}\n' for k, v in zip(keys, vals))
        new, j = vals[-1], int(rng.integers(len(keys)))
        mid = f"\nconfig.update({keys[j]}={new})  # raised after the outage\n"
        query = f'\n# re-apply the current value\nconfig["{keys[j]}"] ='
    return [head, None, mid, None, query], f" {new}"


def gen_increment(rng, n_items: int = 4):
    base = int(rng.integers(100, 900)) * 10 + int(rng.integers(0, 10 - n_items))   # no carry: only the last digit moves
    names = list(rng.choice(MIGRATIONS, size=n_items + 1, replace=False))
    lines = "".join(f'    Migration(id={base + i}, name="{names[i]}"),\n' for i in range(n_items))
    return [None, "\nMIGRATIONS = [\n" + lines + "    Migration(id="], f"{base + n_items}"


def gen_track(rng):
    a = int(rng.integers(0, 6))
    b = int(rng.integers(1, 10 - a))
    var = str(rng.choice(["x", "count", "total", "n", "acc"]))
    return [None, f"\n{var} = {a}\n{var} = {var} + {b}\nprint({var})  # prints"], f" {a + b}"


def make_item(category: str, rng, n_distract: int):
    if category == "copy_same":
        return gen_copy(rng, n_distract, True)
    if category == "copy_diff":
        return gen_copy(rng, n_distract, False)
    if category.startswith("conflict_g"):
        return gen_conflict(rng, n_distract, int(category[len("conflict_g"):]))
    if category == "override_same":
        return gen_override(rng, n_distract, True)
    if category == "override_diff":
        return gen_override(rng, n_distract, False)
    if category == "increment":
        return gen_increment(rng)
    if category == "track":
        return gen_track(rng)
    raise ValueError(category)


def _encoder(tok):
    """Tokenize text as it would appear in the middle of a file (after a newline): SentencePiece otherwise adds a
    word-initial "▁" to every separately encoded part, which changes how the first identifier is split."""
    nl = tok("\n", add_special_tokens=False)["input_ids"]

    def enc(text: str) -> List[int]:
        ids = tok("\n" + text, add_special_tokens=False)["input_ids"]
        return ids[len(nl):] if ids[:len(nl)] == nl else ids
    return enc, len(nl)


def build_items(tok, rng, category: str, n: int, gap_tokens: int, filler: np.ndarray, eos_id: int,
                n_distract: int = 8):
    """-> (seqs, spans, deciding positions). Gaps are filled with real code (EOS removed). The answer span covers
    the digits only (a leading "▁" token, if any, is moved into the prompt); the deciding position is the
    first digit (last digit for `increment`)."""
    enc, n_nl = _encoder(tok)
    nl = tok("\n", add_special_tokens=False)["input_ids"]
    seqs, spans, decide = [], [], []
    for _ in range(n):
        parts, answer = make_item(category, rng, n_distract)
        n_gaps = max(1, sum(p is None for p in parts))
        g = gap_tokens // n_gaps
        ids: List[int] = []
        for p in parts[:-1]:
            if p is None:
                start = int(rng.integers(0, max(1, len(filler) - g - 1)))
                ids += [t for t in np.asarray(filler[start: start + g]).tolist() if t != eos_id]
            else:
                ids += enc(p)
        q_full, (qa, qb) = encode_with_answer(tok, "\n" + parts[-1], answer)
        cut = n_nl if q_full[:n_nl] == nl else 0
        q_ids, qa, qb = q_full[cut:], qa - cut, qb - cut
        while qa < qb - 1 and not tok.decode([q_ids[qa]]).strip():          # whitespace-only token: not an answer
            qa += 1
        a0 = len(ids)
        spans.append((a0 + qa, a0 + qb))
        decide.append(a0 + (qb - 1 if category == "increment" else qa))
        seqs.append(ids + q_ids)
    return seqs, spans, decide


def run_coderecall(model, tok, filler: np.ndarray, eos_id: int, device, amp_dtype,
                   gaps: Sequence[int] = (256, 1024, 3072), n: int = 100, seed: int = 0, n_distract: int = 8,
                   ice_keep: float = 1.0, categories: Optional[Sequence[str]] = None) -> Dict[str, float]:
    res = {}
    for cat in categories or CATEGORIES:
        for gap in (gaps if cat not in ("increment", "track") else gaps[:1]):
            rng = np.random.default_rng([seed, CATEGORIES.index(cat), gap])    # same items for every arm
            seqs, spans, decide = build_items(tok, rng, cat, n, gap, filler, eos_id, n_distract)
            d = score_answers(model, seqs, spans, device, amp_dtype, ice_keep, details=True, decide=decide)
            for k in ("em", "dec_acc", "dec_acc_lookup_right", "dec_acc_lookup_wrong", "dec_acc_lookup_none",
                      "dec_lookup_right_frac", "dec_lookup_wrong_frac", "dec_follow_wrong_lookup"):
                res[f"coderecall/{cat}@{gap}/{k}"] = d[k]
            res[f"coderecall/{cat}@{gap}/items_dec"] = d["items_dec"]          # per item, for paired statistics
    return res


def summarize(res: Dict[str, float], metric: str = "dec_acc") -> Dict[str, float]:
    """Mean of a metric per category (over gaps)."""
    out = {}
    for cat in CATEGORIES:
        v = [x for k, x in res.items() if k.startswith(f"coderecall/{cat}@") and k.endswith(f"/{metric}")
             and not (isinstance(x, float) and np.isnan(x))]
        if v:
            out[cat] = float(np.mean(v))
    return out


In [ ]:
%%writefile ice_lm/ice_lm/inference.py
"""Inference helpers: an incremental ICE index for decoding, and memory accounting.

ICEIndex keeps, per sequence, the token store and one table per order n: n-gram -> (most recent end position,
the one before it). Feeding tokens one at a time yields exactly the features of
ice_features.compute_ice_features (tested in tests/test_ice_lm.py), including the true match length, the
agreement of the two most recent occurrences and the online reliability of each order.

It runs on the CPU, next to the sampling loop; only ~10 small integers per order go to the GPU per step.
Two memory numbers are reported:
  * compact_bytes(): what a C / GPU implementation needs -- token ids (2 B/token for vocab <= 65536) +
    for orders n >= 2 an open-addressing table of 2 x int32 per distinct n-gram at load factor 0.5
    (16 B per distinct n-gram) + a direct int32 [vocab] x 2 table for order 1 + reliability bits.
  * the Python objects used here are much larger (dict + tuple overhead); that is an implementation artefact,
    not a property of the method, and is not what we report as ICE memory.
"""
from __future__ import annotations

from array import array
from collections import deque
from typing import Dict, List, Optional, Sequence

import numpy as np
import torch

from .ice_features import (DEFAULT_ORDERS, MAX_MATCH, REL_EDGES, REL_WINDOW, _dist_bucket, _len_bucket,
                           compute_ice_features, segments_from_eos)

FEATURE_KEYS_N = ("ice_succ", "ice_len", "ice_dist", "ice_agree", "ice_rel")      # [N] per position
FEATURE_KEYS_1 = ("ice_s_star", "ice_len_star", "ice_agree_star", "ice_rel_star", "ice_match_len",
                  "ice_dist_star")                                                 # scalar per position


def _rel_bucket(hist) -> int:
    if not hist:
        return 0
    return 1 + int(np.searchsorted(REL_EDGES, sum(hist) / len(hist), side="right"))


class ICEIndex:
    def __init__(self, orders: Sequence[int] = DEFAULT_ORDERS, eos_id: Optional[int] = None, vocab_size: int = 32000):
        self.orders = tuple(orders)
        self.N = len(self.orders)
        self.eos = eos_id
        self.vocab_size = vocab_size
        self.tokens = array("H" if vocab_size <= 65536 else "I")
        self.tables: List[dict] = [dict() for _ in self.orders]
        self.seg_start = 0
        self.hist = [deque(maxlen=REL_WINDOW) for _ in self.orders]
        self.hist_star = deque(maxlen=REL_WINDOW)
        self.pending = [-1] * self.N
        self.pending_star = -1

    def __len__(self):
        return len(self.tokens)

    def step(self, tok: int) -> Dict[str, object]:
        """Append one token (position t) and return the ICE features of position t (which predicts t+1)."""
        tok = int(tok)
        x = self.tokens
        t = len(x)
        if t > 0 and self.eos is not None and x[t - 1] == self.eos:      # a new document starts at t
            self.seg_start = t
            for h in self.hist:
                h.clear()
            self.hist_star.clear()
            self.pending = [-1] * self.N
            self.pending_star = -1
        # the predictions made at t-1 were about this token: update the reliability history
        for j, pr in enumerate(self.pending):
            if pr >= 0:
                self.hist[j].append(pr == tok)
        if self.pending_star >= 0:
            self.hist_star.append(self.pending_star == tok)
        x.append(tok)

        succ, length, dist, agree = [-1] * self.N, [0] * self.N, [0] * self.N, [0] * self.N
        s0 = self.seg_start
        for j, n in enumerate(self.orders):
            if t - n + 1 < s0:
                continue
            key = tuple(x[t - n + 1: t + 1])
            table = self.tables[j]
            last, second = table.get(key, (-1, -1))
            if last < s0:                     # entries from earlier documents never match
                last = second = -1
            elif second < s0:
                second = -1
            table[key] = (t, last)
            if last < 0:
                continue
            succ[j] = x[last + 1]
            dist[j] = t - last
            k = n                              # the n tokens are equal by construction; extend backwards
            lim = min(MAX_MATCH, last - s0 + 1)
            while k < lim and x[t - k] == x[last - k]:
                k += 1
            length[j] = min(k, MAX_MATCH)
            agree[j] = 1 if second < 0 else (2 if x[second + 1] == x[last + 1] else 3)

        best, best_score = -1, -1
        for j in range(self.N):
            if length[j] > 0 and length[j] * (self.N + 1) + j > best_score:
                best, best_score = j, length[j] * (self.N + 1) + j
        out = dict(
            ice_succ=succ,
            ice_len=_len_bucket(np.array(length)).tolist(),
            ice_dist=_dist_bucket(np.array(dist)).tolist(),
            ice_agree=agree,
            ice_rel=[_rel_bucket(h) for h in self.hist],
            ice_s_star=succ[best] if best >= 0 else -1,
            ice_len_star=int(_len_bucket(np.array([length[best]]))[0]) if best >= 0 else 0,
            ice_agree_star=agree[best] if best >= 0 else 0,
            ice_rel_star=_rel_bucket(self.hist_star),
            ice_match_len=length[best] if best >= 0 else 0,
            ice_dist_star=dist[best] if best >= 0 else 0,
        )
        self.pending = succ
        self.pending_star = out["ice_s_star"]
        return out

    def extend(self, tokens: Sequence[int]) -> Dict[str, np.ndarray]:
        """Feed several tokens; returns features stacked like compute_ice_features ([N,T] and [T])."""
        feats = [self.step(t) for t in tokens]
        if not feats:
            return {**{k: np.zeros((self.N, 0), dtype=np.int64) for k in FEATURE_KEYS_N},
                    **{k: np.zeros(0, dtype=np.int64) for k in FEATURE_KEYS_1}}
        out = {k: np.array([f[k] for f in feats], dtype=np.int64).T for k in FEATURE_KEYS_N}
        out.update({k: np.array([f[k] for f in feats], dtype=np.int64) for k in FEATURE_KEYS_1})
        return out

    def prefill(self, tokens: Sequence[int]) -> Dict[str, np.ndarray]:
        """Process a whole prompt at once (empty index only): features from the vectorized compute_ice_features,
        then the tables and reliability history that step() needs to continue. Same features, tables and state
        as extend() (tables keep earlier documents' n-grams too, as a simple implementation would)."""
        if len(self):
            raise ValueError("prefill() needs an empty index; use extend() to add tokens")
        x = np.asarray(tokens, dtype=np.int64)
        T = len(x)
        if T == 0:
            return self.extend([])
        seg = segments_from_eos(x, self.eos)
        f = compute_ice_features(x, seg, self.orders)
        xl = x.tolist()
        self.tokens.extend(xl)
        s0 = int(np.searchsorted(seg, seg[-1]))              # start of the current (last) document
        self.seg_start = s0
        segl = seg.tolist()
        for j, n in enumerate(self.orders):
            table = self.tables[j]
            for t in range(n - 1, T):
                if segl[t - n + 1] != segl[t]:                  # n-gram crosses a document boundary
                    continue
                key = tuple(xl[t - n + 1: t + 1])
                prev = table.get(key)
                table[key] = (t, prev[0] if prev and segl[prev[0]] == segl[t] else -1)

        def history(pred):
            ev = np.nonzero(pred[s0: T - 1] >= 0)[0] + s0      # predictions whose outcome is known
            return deque((pred[ev] == x[ev + 1]).tolist()[-REL_WINDOW:], maxlen=REL_WINDOW)

        self.hist = [history(f["ice_succ"][j]) for j in range(self.N)]
        self.hist_star = history(f["ice_s_star"])
        self.pending = [int(v) for v in f["ice_succ"][:, T - 1]]
        self.pending_star = int(f["ice_s_star"][T - 1])
        return f

    # -- memory --------------------------------------------------------------
    def distinct_ngrams(self) -> List[int]:
        return [len(t) for t in self.tables]

    def compact_bytes(self) -> Dict[str, int]:
        tok_b = len(self.tokens) * (2 if self.vocab_size <= 65536 else 4)
        tables = 0
        for n, d in zip(self.orders, self.distinct_ngrams()):
            tables += self.vocab_size * 8 if n == 1 else d * 16     # 2 x int32 per entry; load factor 0.5
        rel = (self.N + 1) * REL_WINDOW // 8
        return dict(tokens=tok_b, tables=tables, reliability=rel, total=tok_b + tables + rel)


def features_to_tensors(feats: List[Dict[str, np.ndarray]], device) -> Dict[str, torch.Tensor]:
    """Batch the per-row features of a chunk ([N,T] / [T] arrays) into model inputs [B,N,T] / [B,T]."""
    keys = FEATURE_KEYS_N + FEATURE_KEYS_1
    return {k: torch.from_numpy(np.stack([np.asarray(f[k], dtype=np.int64) for f in feats])).to(device)
            for k in keys}


def step_features_to_tensors(feats: List[Dict[str, object]], device) -> Dict[str, torch.Tensor]:
    """Batch single-position features (ICEIndex.step outputs) into [B,N,1] / [B,1] tensors."""
    out = {k: torch.tensor([f[k] for f in feats], dtype=torch.long)[:, :, None] for k in FEATURE_KEYS_N}
    out.update({k: torch.tensor([f[k] for f in feats], dtype=torch.long)[:, None] for k in FEATURE_KEYS_1})
    return {k: v.to(device) for k, v in out.items()}


In [ ]:
%%writefile ice_lm/ice_lm/utils.py
"""Distributed helpers, precision selection, optimizer, LR schedule, checkpoints, logging."""
from __future__ import annotations

import datetime
import json
import math
import os
import time
import uuid
from typing import Optional

import torch
import torch.distributed as dist


def setup_distributed():
    """Returns (rank, world_size, local_rank, device). Works with or without torchrun."""
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ and int(os.environ["WORLD_SIZE"]) > 1:
        backend = "nccl" if torch.cuda.is_available() else "gloo"
        # generous timeout: the other ranks wait while rank 0 runs the periodic evaluation alone
        dist.init_process_group(backend, timeout=datetime.timedelta(minutes=60))
        rank, world = dist.get_rank(), dist.get_world_size()
        local = int(os.environ.get("LOCAL_RANK", 0))
        if torch.cuda.is_available():
            torch.cuda.set_device(local)
            device = torch.device("cuda", local)
        else:
            device = torch.device("cpu")
        return rank, world, local, device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return 0, 1, 0, device


def is_main(rank: int) -> bool:
    return rank == 0


def cleanup_distributed():
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()


def pick_precision(requested: str, device: torch.device):
    """-> (amp_dtype or None, use_grad_scaler)."""
    if device.type != "cuda" or requested == "fp32":
        return None, False
    if requested == "bf16" or (requested == "auto" and torch.cuda.is_bf16_supported()):
        # T4 / P100 report bf16 as "supported" only via slow emulation on some torch versions
        major, _ = torch.cuda.get_device_capability(device)
        if major >= 8 or requested == "bf16":
            return torch.bfloat16, False
    return torch.float16, True


def build_optimizer(model, lr, weight_decay, betas, device, zero: bool = False):
    """AdamW; zero=True shards the Adam states over the DDP ranks (ZeRO-1: 0.6B on 2 GPUs saves ~2.4 GB each)."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        ice_table = name.startswith("ice_") and name.split(".")[-2] in ("L", "D", "b", "A", "R", "a", "r")
        if p.ndim < 2 or getattr(p, "_no_weight_decay", False) or "norm" in name or "embed" in name or ice_table:
            no_decay.append(p)
        else:
            decay.append(p)
    groups = [dict(params=decay, weight_decay=weight_decay), dict(params=no_decay, weight_decay=0.0)]
    kw = dict(lr=lr, betas=betas)
    if device.type == "cuda":
        kw["fused"] = True
    def make(**kw):
        if zero:
            from torch.distributed.optim import ZeroRedundancyOptimizer
            return ZeroRedundancyOptimizer(groups, optimizer_class=torch.optim.AdamW, **kw)
        return torch.optim.AdamW(groups, **kw)

    try:
        return make(**kw)
    except (TypeError, RuntimeError):
        kw.pop("fused", None)
        return make(**kw)


def lr_at(step: int, total_steps: int, warmup_steps: int, lr: float, min_ratio: float) -> float:
    if step < warmup_steps:
        return lr * (step + 1) / max(1, warmup_steps)
    prog = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
    return lr * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * prog)))


def save_checkpoint(path: str, model, optimizer, scaler, state: dict):
    tmp = path + ".tmp"
    raw = model.module if hasattr(model, "module") else model
    torch.save(dict(model=raw.state_dict(), optimizer=optimizer.state_dict() if optimizer else None,
                    scaler=scaler.state_dict() if scaler is not None else None, state=state), tmp)
    os.replace(tmp, path)


def load_checkpoint(path: str, model, optimizer=None, scaler=None, map_location="cpu") -> dict:
    ck = torch.load(path, map_location=map_location, weights_only=False)
    raw = model.module if hasattr(model, "module") else model
    try:
        raw.load_state_dict(ck["model"])
    except RuntimeError as e:
        raise RuntimeError(f"checkpoint {path} does not fit this model (different preset/arm/vocab?). "
                           f"Use a new out_dir or delete the old checkpoint.\n{e}") from None
    if optimizer is not None and ck.get("optimizer") is not None:
        optimizer.load_state_dict(ck["optimizer"])
    if scaler is not None and ck.get("scaler") is not None:
        scaler.load_state_dict(ck["scaler"])
    return ck.get("state", {})


def _shard_path(path: str, rank: int, world: int) -> str:
    return f"{path}.optim{rank}of{world}"


def save_optimizer_shard(path: str, optimizer, rank: int, world: int, step: int):
    """ZeRO: every rank saves its own optimizer partition (no gather of the full state onto one GPU)."""
    tmp = _shard_path(path, rank, world) + ".tmp"
    torch.save(dict(optim=optimizer.optim.state_dict(), step=step), tmp)
    os.replace(tmp, _shard_path(path, rank, world))


def load_optimizer_shard(path: str, optimizer, rank: int, world: int, step: int, map_location="cpu") -> bool:
    p = _shard_path(path, rank, world)
    if not os.path.exists(p):
        return False
    ck = torch.load(p, map_location=map_location, weights_only=False)
    if ck.get("step") != step:
        return False
    optimizer.optim.load_state_dict(ck["optim"])
    return True


def init_wandb(out_dir: str, name: str, config: dict):
    """Weights & Biases run when WANDB_API_KEY is set (e.g. from a Modal secret), else None. The run id is kept in
    out_dir/wandb_id.txt, so a resumed training run continues the same W&B run. Any W&B error disables it."""
    if not os.environ.get("WANDB_API_KEY"):
        return None
    try:
        import wandb
        id_path = os.path.join(out_dir, "wandb_id.txt")
        run_id = open(id_path).read().strip() if os.path.exists(id_path) else uuid.uuid4().hex[:12]
        with open(id_path, "w") as f:
            f.write(run_id)
        return wandb.init(project=os.environ.get("WANDB_PROJECT", "ice-lm"), name=name, id=run_id, resume="allow",
                          config=config, dir=out_dir)
    except Exception as e:  # noqa: BLE001 - logging must never stop training
        print(f"[wandb] disabled ({type(e).__name__}: {e})")
        return None


class JsonlLogger:
    def __init__(self, path: Optional[str], wandb_run=None):
        self.path = path
        self.wandb = wandb_run

    def log(self, **kv):
        kv["time"] = time.time()
        if self.path:
            with open(self.path, "a") as f:
                f.write(json.dumps(kv) + "\n")
        if self.wandb is not None:
            try:
                self.wandb.log({k: v for k, v in kv.items() if k not in ("step", "time")}, step=kv.get("step"))
            except Exception as e:  # noqa: BLE001
                print(f"[wandb] log failed, disabling ({type(e).__name__}: {e})")
                self.wandb = None

    def finish(self):
        if self.wandb is not None:
            self.wandb.finish()


In [ ]:
%%writefile ice_lm/scripts/prepare_data.py
#!/usr/bin/env python
"""Download + tokenize the training mixture (default 70% FineWeb-Edu prose, 30% Python code).

    python scripts/prepare_data.py --out data --train_tokens 2.5e9
    python scripts/prepare_data.py --out data_synth --synthetic          # offline toy corpus for smoke tests
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from ice_lm.data import TOKENIZER_DEFAULT, make_synthetic_data, prepare_data  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--out", default="data")
ap.add_argument("--train_tokens", type=float, default=2.5e9)
ap.add_argument("--val_tokens", type=float, default=2e6, help="per validation split (prose and code)")
ap.add_argument("--code_frac", type=float, default=0.3)
ap.add_argument("--tokenizer", default=TOKENIZER_DEFAULT)
ap.add_argument("--prose_dataset", default="HuggingFaceFW/fineweb-edu")
ap.add_argument("--prose_config", default="sample-10BT")
ap.add_argument("--prose_field", default="text")
ap.add_argument("--code_dataset", default="codeparrot/codeparrot-clean")
ap.add_argument("--code_config", default=None)
ap.add_argument("--code_field", default="content")
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--synthetic", action="store_true")
a = ap.parse_args()

if a.synthetic:
    print(make_synthetic_data(a.out, train_tokens=int(a.train_tokens) if a.train_tokens < 1e8 else 3_000_000,
                              seed=a.seed))
else:
    prepare_data(a.out, int(a.train_tokens), int(a.val_tokens), a.code_frac, a.tokenizer,
                 prose=dict(path=a.prose_dataset, name=a.prose_config, field=a.prose_field),
                 code=dict(path=a.code_dataset, name=a.code_config, field=a.code_field), seed=a.seed)

# streaming datasets leave download threads behind that can block interpreter shutdown (seen on Kaggle);
# everything is written and closed at this point, so exit right away
sys.stdout.flush()
sys.stderr.flush()
os._exit(0)


In [ ]:
%%writefile ice_lm/scripts/day0_stats.py
#!/usr/bin/env python
"""Day 0 (proposal, Step 0 ceiling): no model needed.
How often does an earlier n-gram match exist (coverage), and how often is its successor the true next
token (precision)? Reported per order and by longest-match length, separately for prose and code.
Also: is the lookup's precision predictable from the "trust" signals the model gets (agreement of the two
most recent occurrences, online reliability of the longest-match predictor)? If precision differs a lot
across their buckets, the model has something to learn from when deciding to ignore a lookup.

    python scripts/day0_stats.py --data_dir data --seq_len 4096 --windows 200
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import numpy as np  # noqa: E402

from ice_lm.data import load_meta, open_tokens  # noqa: E402
from ice_lm.ice_features import DEFAULT_ORDERS, compute_ice_features, segments_from_eos  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--data_dir", default="data")
ap.add_argument("--seq_len", type=int, default=4096)
ap.add_argument("--windows", type=int, default=200)
ap.add_argument("--out", default=None, help="optional JSON output path")
a = ap.parse_args()

meta = load_meta(a.data_dir)
LEN_BINS = [(1, 1), (2, 3), (4, 7), (8, 15), (16, 31), (32, 32)]
AGREE_NAMES = ["none", "single", "agree", "disagree"]
REL_NAMES = ["no history", "<20%", "20-50%", "50-80%", ">=80%"]
DIST_BINS = [(1, 64), (65, 512), (513, 2048), (2049, 1 << 30)]       # how far back the longest match is
results = {}
for split in ("val_prose", "val_code"):
    data = open_tokens(a.data_dir, split)
    n_win = min(a.windows, (len(data) - 1) // a.seq_len)
    order_hit = np.zeros(len(DEFAULT_ORDERS)); order_cov = np.zeros(len(DEFAULT_ORDERS))
    len_hit = np.zeros(len(LEN_BINS)); len_cnt = np.zeros(len(LEN_BINS))
    ag_hit = np.zeros(4); ag_cnt = np.zeros(4); rel_hit = np.zeros(5); rel_cnt = np.zeros(5)
    d_hit = np.zeros(len(DIST_BINS)); d_cnt = np.zeros(len(DIST_BINS))
    total = 0
    for j in range(n_win):
        w = np.asarray(data[j * a.seq_len: j * a.seq_len + a.seq_len + 1]).astype(np.int64)
        x, y = w[:-1], w[1:]
        f = compute_ice_features(x, segments_from_eos(x, meta["eos_id"]))
        total += len(x)
        for k in range(len(DEFAULT_ORDERS)):
            has = f["ice_succ"][k] >= 0
            order_cov[k] += has.sum()
            order_hit[k] += (f["ice_succ"][k][has] == y[has]).sum()
        ml = f["ice_match_len"]
        for b, (lo, hi) in enumerate(LEN_BINS):
            sel = (ml >= lo) & (ml <= hi)
            len_cnt[b] += sel.sum()
            len_hit[b] += (f["ice_s_star"][sel] == y[sel]).sum()
        has = f["ice_s_star"] >= 0
        right = (f["ice_s_star"] == y) & has
        for b in range(4):
            sel = has & (f["ice_agree_star"] == b)
            ag_cnt[b] += sel.sum(); ag_hit[b] += right[sel].sum()
        for b in range(5):
            sel = has & (f["ice_rel_star"] == b)
            rel_cnt[b] += sel.sum(); rel_hit[b] += right[sel].sum()
        for b, (lo, hi) in enumerate(DIST_BINS):
            sel = has & (f["ice_dist_star"] >= lo) & (f["ice_dist_star"] <= hi)
            d_cnt[b] += sel.sum(); d_hit[b] += right[sel].sum()
    res = dict(tokens=int(total),
               per_order={str(n): dict(coverage=order_cov[k] / total, precision=order_hit[k] / max(1, order_cov[k]))
                          for k, n in enumerate(DEFAULT_ORDERS)},
               longest_match={f"{lo}-{hi}": dict(coverage=len_cnt[b] / total, precision=len_hit[b] / max(1, len_cnt[b]))
                              for b, (lo, hi) in enumerate(LEN_BINS)})
    res["ice_correct_frac"] = float(len_hit.sum() / total)
    res["by_agree"] = {AGREE_NAMES[b]: dict(coverage=ag_cnt[b] / total, precision=ag_hit[b] / max(1, ag_cnt[b]))
                       for b in range(1, 4)}
    res["by_distance"] = {(f"{lo}-{hi}" if hi < 1 << 30 else f">{lo - 1}"):
                          dict(coverage=d_cnt[b] / total, precision=d_hit[b] / max(1, d_cnt[b]),
                               correct_frac=d_hit[b] / total)
                          for b, (lo, hi) in enumerate(DIST_BINS)}
    res["by_reliability"] = {REL_NAMES[b]: dict(coverage=rel_cnt[b] / total, precision=rel_hit[b] / max(1, rel_cnt[b]))
                             for b in range(5)}
    results[split] = res
    print(f"\n=== {split}: {total:,} tokens ===")
    print(" order  coverage  precision")
    for n, r in res["per_order"].items():
        print(f"  {n:>4}   {r['coverage']:7.1%}   {r['precision']:7.1%}")
    print(" longest match  coverage  precision")
    for k, r in res["longest_match"].items():
        print(f"  {k:>11}   {r['coverage']:7.1%}   {r['precision']:7.1%}")
    print(f" tokens predicted correctly by the longest-match successor: {res['ice_correct_frac']:.1%}")
    print(" trust signal of the longest match       coverage  precision")
    for k, r in res["by_agree"].items():
        print(f"  occurrences: {k:<24} {r['coverage']:7.1%}   {r['precision']:7.1%}")
    for k, r in res["by_reliability"].items():
        print(f"  recent hit rate: {k:<20} {r['coverage']:7.1%}   {r['precision']:7.1%}")
    print(" distance of the longest match           coverage  precision  tokens it gets right")
    for k, r in res["by_distance"].items():
        print(f"  {k:<38} {r['coverage']:7.1%}   {r['precision']:7.1%}   {r['correct_frac']:7.1%}")
if a.out:
    with open(a.out, "w") as fh:
        json.dump(results, fh, indent=2, default=float)


In [ ]:
%%writefile ice_lm/scripts/train.py
#!/usr/bin/env python
"""Train one arm.  Single GPU:  python scripts/train.py --preset 0.6b --arm C --data_dir data --out_dir runs/C
Multi-GPU (e.g. Kaggle 2xT4):  torchrun --nproc_per_node=2 scripts/train.py ...
Re-running the same command resumes from out_dir/ckpt_latest.pt (if present)."""
from __future__ import annotations

import argparse
import dataclasses
import json
import math
import os
import sys
import time
from contextlib import nullcontext

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation on 16 GB GPUs

import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader

from ice_lm.config import TrainConfig, ModelConfig, build_configs, dump_json, model_config_from_dict, to_dict
from ice_lm.data import TrainWindows, collate, load_meta, split_batch
from ice_lm.evals import eval_loss_splits
from ice_lm.model import ICELM, kv_bytes_per_token, layer_plan
from ice_lm.utils import (JsonlLogger, build_optimizer, cleanup_distributed, init_wandb, is_main, load_checkpoint,
                          load_optimizer_shard, lr_at, pick_precision, save_checkpoint, save_optimizer_shard,
                          setup_distributed)


def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--preset", default="140m")
    ap.add_argument("--arm", default="A")
    tfields = {f.name: f for f in dataclasses.fields(TrainConfig)}
    mfields = {f.name: f for f in dataclasses.fields(ModelConfig)}
    for name, f in {**mfields, **tfields}.items():
        if name in ("preset", "arm", "ice_orders"):
            continue
        typ = f.type if isinstance(f.type, type) else eval(f.type)  # noqa: S307 - dataclass annotations
        if typ is bool:
            ap.add_argument(f"--{name}", type=lambda s: s.lower() in ("1", "true", "yes", "y"), default=None)
        else:
            conv = float if typ is float else ((lambda v: int(float(v))) if typ is int else str)
            ap.add_argument(f"--{name}", type=conv, default=None)
    return ap.parse_args()


# settings that define the learning-rate schedule and the data: a resumed run keeps its saved values.
# (micro_batch / num_workers / GPU count may change: the samples of optimizer step s are always the indices
#  [s*S, (s+1)*S) with S = global batch / seq_len, see ice_lm/data.py TrainWindows)
RESUME_KEYS = ("total_tokens", "global_batch_tokens", "lr", "min_lr_ratio", "warmup_tokens", "seq_len", "seed",
               "ice_dropout", "weight_decay", "beta1", "beta2", "grad_clip")


def main():
    args = vars(parse_args())
    preset, arm = args.pop("preset"), args.pop("arm")
    rank, world, local, device = setup_distributed()
    meta = load_meta(args.get("data_dir") or "data")
    args["vocab_size"] = args.get("vocab_size") or meta["vocab_size"]
    mcfg, tcfg = build_configs(preset, arm, **args)
    torch.manual_seed(tcfg.seed)
    main_proc = is_main(rank)
    os.makedirs(tcfg.out_dir, exist_ok=True)
    if tcfg.resume and os.path.exists(os.path.join(tcfg.out_dir, "model_final.pt")):
        if main_proc:
            print(f"[train] {tcfg.out_dir} is already finished (model_final.pt); nothing to do "
                  f"(use a new out_dir, or --resume false to retrain from scratch)")
        cleanup_distributed()
        return
    saved_cfg = os.path.join(tcfg.out_dir, "config.json")
    data_meta = meta
    if tcfg.resume and os.path.exists(os.path.join(tcfg.out_dir, "ckpt_latest.pt")) and os.path.exists(saved_cfg):
        # continuing a run: keep the architecture and the schedule / data order it was started with
        with open(saved_cfg) as f:
            saved = json.load(f)
        old = model_config_from_dict(saved["model"])
        if tcfg.keep_resume_config:
            fixed = {k: v for k, v in saved.get("train", {}).items() if k in RESUME_KEYS and getattr(tcfg, k) != v}
            if fixed:
                if main_proc:
                    print(f"[train] resuming with the run's saved settings (ignoring the new values): "
                          f"{ {k: (v, getattr(tcfg, k)) for k, v in fixed.items()} }  [saved, given]")
                tcfg = dataclasses.replace(tcfg, **fixed)
        old_fp = saved.get("data_meta", {}).get("fingerprint")
        if old_fp and meta.get("fingerprint") and old_fp != meta["fingerprint"] and main_proc:
            print(f"[train] WARNING: the data differs from the data this run started with (fingerprint "
                  f"{meta['fingerprint']} vs {old_fp}), e.g. a re-download in a new session: the comparison with "
                  f"other arms is no longer exact")
        data_meta = saved.get("data_meta", meta)         # keep the original record
        if old != mcfg:
            if main_proc:
                diff = {k: (getattr(old, k), getattr(mcfg, k)) for k in to_dict(mcfg) if getattr(old, k) != getattr(mcfg, k)}
                print(f"[train] resuming with the run's saved model config (differs from the current defaults: {diff})")
            mcfg = dataclasses.replace(old, grad_checkpointing=mcfg.grad_checkpointing, gdn_backend=mcfg.gdn_backend)

    amp_dtype, use_scaler = pick_precision(tcfg.precision, device)
    model = ICELM(mcfg).to(device)
    model.prepare_backend(device, amp_dtype or torch.float32)
    if main_proc:
        plan = layer_plan(mcfg)
        print(f"[train] preset={preset} arm={arm} device={device} world={world} amp={amp_dtype} scaler={use_scaler}")
        print(f"[train] params total={model.num_params()/1e6:.1f}M  non-embedding={model.num_params(True)/1e6:.1f}M"
              f"  ICE={model.ice_params()/1e6:.2f}M")
        print(f"[train] layout: " + " ".join("A" if p['gdn'] == 0 else ("G" if p['attn'] == 0 else "h") for p in plan)
              + f"   (A=attention, G=GDN, h=mixed heads)   KV bytes/token={kv_bytes_per_token(mcfg)}")
        dump_json(os.path.join(tcfg.out_dir, "config.json"),
                  dict(model=to_dict(mcfg), train=to_dict(tcfg), data_meta=data_meta, world=world))

    tokens_per_micro = tcfg.micro_batch * tcfg.seq_len * world
    grad_accum = max(1, round(tcfg.global_batch_tokens / tokens_per_micro))
    if main_proc and tokens_per_micro * grad_accum != tcfg.global_batch_tokens:
        print(f"[train] WARNING: global batch {tcfg.global_batch_tokens:,} tokens is not a multiple of "
              f"micro_batch*seq_len*GPUs = {tokens_per_micro:,}; using {tokens_per_micro * grad_accum:,} "
              f"(runs with different settings will not see the same data per step)")
    tokens_per_step = tokens_per_micro * grad_accum
    total_steps = max(1, math.ceil(tcfg.total_tokens / tokens_per_step))
    warmup_steps = max(1, round(tcfg.warmup_tokens / tokens_per_step))
    zero = world > 1 and tcfg.zero_optimizer
    opt = build_optimizer(model, tcfg.lr, tcfg.weight_decay, (tcfg.beta1, tcfg.beta2), device, zero=zero)
    scaler = torch.amp.GradScaler("cuda") if use_scaler else None

    state = dict(step=0, tokens=0, samples=0, best_val=None)
    ckpt_path = os.path.join(tcfg.out_dir, "ckpt_latest.pt")
    if tcfg.resume and os.path.exists(ckpt_path):
        state = load_checkpoint(ckpt_path, model, None if zero else opt, scaler, map_location=device)
        opt_ok = True
        if zero:
            opt_ok = load_optimizer_shard(ckpt_path, opt, rank, world, state["step"], map_location=device)
            flag = torch.tensor([float(opt_ok)], device=device)
            dist.all_reduce(flag, op=dist.ReduceOp.MIN)
            if not bool(flag[0] > 0):     # some rank is missing its shard: fresh optimizer on all ranks
                opt = build_optimizer(model, tcfg.lr, tcfg.weight_decay, (tcfg.beta1, tcfg.beta2), device, zero=zero)
            opt_ok = bool(flag[0] > 0)
        if not zero:
            opt_ok = len(opt.state) > 0          # False e.g. for a ZeRO checkpoint (sharded optimizer) on 1 GPU
        if main_proc:
            print(f"[train] resumed from {ckpt_path} at step {state['step']} ({state['tokens']/1e6:.1f}M tokens)"
                  + ("" if opt_ok else "  [optimizer state not found for this GPU count: Adam moments reset]"))
    if state["step"] >= total_steps:
        if main_proc:
            print("[train] already finished")
        cleanup_distributed()
        return

    if tcfg.compile:
        model = torch.compile(model)
    # gradient_as_bucket_view: no second copy of the gradients (matters on 16 GB GPUs)
    ddp_model = (DDP(model, device_ids=[local] if device.type == "cuda" else None, gradient_as_bucket_view=True)
                 if world > 1 else model)
    raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model

    with_ice = mcfg.ice
    ds = TrainWindows(tcfg.data_dir, tcfg.seq_len, mcfg.ice_orders, with_ice, tcfg.ice_dropout, tcfg.seed,
                      rank, world, start_index=state.get("samples", state["step"] * tokens_per_step // tcfg.seq_len),
                      micro_batch=tcfg.micro_batch)
    loader = DataLoader(ds, batch_size=tcfg.micro_batch, collate_fn=collate, num_workers=tcfg.num_workers,
                        pin_memory=device.type == "cuda", persistent_workers=tcfg.num_workers > 0,
                        prefetch_factor=4 if tcfg.num_workers > 0 else None)
    it = iter(loader)
    wb = (init_wandb(tcfg.out_dir, os.path.basename(os.path.normpath(tcfg.out_dir)),
                     dict(model=to_dict(mcfg), train=to_dict(tcfg), data_meta=data_meta, world=world,
                          params=model.num_params(), ice_params=model.ice_params()))
          if main_proc else None)
    logger = JsonlLogger(os.path.join(tcfg.out_dir, "log.jsonl") if main_proc else None, wb)
    autocast = (lambda: torch.autocast("cuda", dtype=amp_dtype)) if amp_dtype else nullcontext

    if main_proc:
        print(f"[train] steps={total_steps} (now {state['step']})  grad_accum={grad_accum}  "
              f"tokens/step={tokens_per_step:,}  warmup_steps={warmup_steps}")
    t_start = time.time()
    t_ckpt = time.time()
    t_log, tok_log = time.time(), 0
    ddp_model.train()
    stop_reason = None
    while state["step"] < total_steps:
        step = state["step"]
        lr = lr_at(step, total_steps, warmup_steps, tcfg.lr, tcfg.min_lr_ratio)
        for gparam in opt.param_groups:
            gparam["lr"] = lr
        loss_acc = torch.zeros((), device=device)      # accumulated on the GPU: no sync per micro-step
        for micro in range(grad_accum):
            batch = next(it)
            x, y, ice, keep, _ = split_batch(batch, device)
            sync = nullcontext() if (world == 1 or micro == grad_accum - 1) else ddp_model.no_sync()
            with sync:
                with autocast():
                    _, loss = ddp_model(x, y, ice=ice, ice_keep=keep)
                loss = loss / grad_accum
                (scaler.scale(loss) if scaler else loss).backward()
            loss_acc += loss.detach().float()
        if scaler:
            scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), tcfg.grad_clip)
        if scaler:
            scaler.step(opt)
            scaler.update()
        else:
            opt.step()
        opt.zero_grad(set_to_none=True)
        state["step"] += 1
        state["tokens"] += tokens_per_step
        state["samples"] += tcfg.micro_batch * grad_accum * world
        tok_log += tokens_per_step

        if state["step"] % tcfg.log_every == 0 or state["step"] == 1:      # same decision on every rank
            if world > 1:
                dist.all_reduce(loss_acc, op=dist.ReduceOp.SUM)  # SUM works on NCCL and gloo
            loss_val, gnorm = loss_acc.item() / world, float(gnorm)
            if main_proc:
                dt = time.time() - t_log
                tps = tok_log / max(dt, 1e-9)
                eta_h = (total_steps - state["step"]) * tokens_per_step / max(tps, 1e-9) / 3600
                extra = f"  scale {scaler.get_scale():.0f}" if scaler else ""
                print(f"step {state['step']:>6}/{total_steps}  loss {loss_val:.4f}  lr {lr:.2e}  gnorm {gnorm:.2f}  "
                      f"tok/s {tps:,.0f}  tokens {state['tokens']/1e6:,.1f}M  eta {eta_h:.1f}h{extra}", flush=True)
                logger.log(step=state["step"], loss=loss_val, lr=lr, gnorm=gnorm, tok_per_s=tps, tokens=state["tokens"],
                           **({"loss_scale": scaler.get_scale()} if scaler else {}))
                t_log, tok_log = time.time(), 0

        if tcfg.eval_every and state["step"] % tcfg.eval_every == 0 and main_proc:
            res = eval_loss_splits(raw_model, tcfg.data_dir, tcfg.seq_len, tcfg.micro_batch, tcfg.eval_batches,
                                   device, amp_dtype)
            print("[eval] " + "  ".join(f"{k}={v:.4f}" for k, v in res.items()
                                        if k.split("/")[-1] in ("all_loss", "copyable_loss", "lk_wrong_loss",
                                                                 "follow_wrong_lookup")), flush=True)
            logger.log(step=state["step"], tokens=state["tokens"], **{f"val/{k}": v for k, v in res.items()})

        # stop / checkpoint decisions must be identical on all ranks
        flags = torch.tensor([float((time.time() - t_start) / 3600 >= tcfg.max_hours),
                              float((time.time() - t_ckpt) / 60 >= tcfg.ckpt_every_minutes)], device=device)
        if world > 1:
            dist.all_reduce(flags, op=dist.ReduceOp.MAX)
        if flags[0] > 0:
            stop_reason = f"max_hours={tcfg.max_hours} reached"
        time_for_ckpt = bool(flags[1] > 0)
        finishing = state["step"] >= total_steps and not tcfg.keep_checkpoint    # model_final.pt follows
        if (stop_reason or time_for_ckpt or state["step"] >= total_steps) and not finishing:
            if zero:
                save_optimizer_shard(ckpt_path, opt, rank, world, state["step"])
            if main_proc:
                save_checkpoint(ckpt_path, raw_model, None if zero else opt, scaler, state)
                print(f"[train] checkpoint saved at step {state['step']}", flush=True)
            t_ckpt = time.time()
            if world > 1:
                dist.barrier()
        if stop_reason:
            break

    if main_proc:
        if state["step"] >= total_steps:
            final = os.path.join(tcfg.out_dir, "model_final.pt")
            torch.save(dict(model=raw_model.state_dict(), model_config=to_dict(mcfg), state=state), final + ".tmp")
            os.replace(final + ".tmp", final)
            # the run is complete: the resume checkpoint + optimizer shards are no longer needed (Kaggle disk is
            # ~20 GB; for 0.6b they are ~7 GB). Set keep_checkpoint to keep them.
            if not tcfg.keep_checkpoint:
                for f in os.listdir(tcfg.out_dir):
                    if f.startswith("ckpt_latest.pt"):
                        os.remove(os.path.join(tcfg.out_dir, f))
            print(f"[train] finished: {state['tokens']/1e6:,.1f}M tokens -> {final}")
        else:
            print(f"[train] stopped ({stop_reason}); re-run the same command to resume")
        logger.finish()
    cleanup_distributed()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ice_lm/scripts/evaluate.py
#!/usr/bin/env python
"""Evaluate trained arms and print comparison tables.

    python scripts/evaluate.py --runs runs/A runs/B runs/C --data_dir data --out results.json
Uses model_final.pt if present, otherwise ckpt_latest.pt.

For ICE arms the same evaluations are repeated with ICE switched off ("<arm>_off", keep = 0): if C_off is worse
than B, the model has become lazy (relies on the lookup instead of learning to recall). This control is only
meaningful for arms trained with ICE-dropout (not C_nodrop, for which keep = 0 is out of distribution).

CodeRecall is reported on the deciding digit (see ice_lm/coderecall.py) with paired bootstrap 95% intervals
for the difference to the reference arm (B if present): every arm scores identical items.
"""
import argparse
import json
import math
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import numpy as np  # noqa: E402
import torch  # noqa: E402

from ice_lm.coderecall import CATEGORIES, run_coderecall, summarize  # noqa: E402
from ice_lm.config import model_config_from_dict  # noqa: E402
from ice_lm.data import load_meta, open_tokens  # noqa: E402
from ice_lm.evals import eval_loss_splits, run_synthetic_tasks, uses_ice  # noqa: E402
from ice_lm.model import ICELM, kv_bytes_per_token  # noqa: E402
from ice_lm.utils import pick_precision  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--runs", nargs="*", default=[])
ap.add_argument("--from_json", nargs="*", default=[], help="merge results saved by earlier evaluate.py calls "
                "(no model evaluation for those; only the comparison tables)")
ap.add_argument("--data_dir", default="data")
ap.add_argument("--seq_len", type=int, default=None, help="default: training seq_len")
ap.add_argument("--val_batches", type=int, default=50)
ap.add_argument("--batch_size", type=int, default=1)
ap.add_argument("--context_lens", type=int, nargs="+", default=[1024, 4096],
                help="lengths above the training seq_len test extrapolation (attention degrades, ICE does not)")
ap.add_argument("--n_task", type=int, default=50, help="examples per synthetic task and length")
ap.add_argument("--coderecall_gaps", type=int, nargs="+", default=[256, 1024, 3072],
                help="total filler tokens per item; keep below the training seq_len")
ap.add_argument("--n_coderecall", type=int, default=100, help="items per CodeRecall category and gap")
ap.add_argument("--reference", default="B", help="arm used for paired differences / excess follow rate")
ap.add_argument("--no_ice_off", action="store_true", help="skip the ICE-switched-off control")
ap.add_argument("--precision", default="auto")
ap.add_argument("--out", default="results.json")
a = ap.parse_args()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype, _ = pick_precision(a.precision, device)
meta = load_meta(a.data_dir)
tok = None
if meta.get("tokenizer"):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(meta["tokenizer"])


def evaluate(model, seq_len, keep):
    res = eval_loss_splits(model, a.data_dir, seq_len, a.batch_size, a.val_batches, device, amp_dtype,
                           ice_keep=keep)
    res.update(run_synthetic_tasks(model, a.data_dir, device, amp_dtype, tuple(a.context_lens), a.n_task,
                                   ice_keep=keep))
    if tok is not None:
        res.update(run_coderecall(model, tok, open_tokens(a.data_dir, "val_code"), meta["eos_id"], device,
                                  amp_dtype, tuple(a.coderecall_gaps), a.n_coderecall, ice_keep=keep))
    return res


prints = {json.load(open(os.path.join(r, "config.json"))).get("data_meta", {}).get("fingerprint") for r in a.runs}
if not a.runs and not a.from_json:
    ap.error("give --runs and/or --from_json")
if len(prints - {None}) > 1:
    print(f"WARNING: these runs were trained on different data (fingerprints {sorted(prints - {None})}): "
          f"the comparison is not fair")

all_res = {}
for fj in a.from_json:
    all_res.update(json.load(open(fj)))
for run in a.runs:
    cfg = json.load(open(os.path.join(run, "config.json")))
    mcfg = model_config_from_dict(cfg["model"])
    model = ICELM(mcfg).to(device)
    path = os.path.join(run, "model_final.pt")
    if not os.path.exists(path):
        path = os.path.join(run, "ckpt_latest.pt")
    ck = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    model.prepare_backend(device, amp_dtype or torch.float32)
    seq_len = a.seq_len or cfg["train"]["seq_len"]
    name = cfg["train"]["arm"]
    if mcfg.ice and not mcfg.ice_trust and "notrust" not in name:
        name += "(v1)"                       # trained before the trust features existed
    tokens = ck.get("state", {}).get("tokens", 0)
    print(f"\n### {name} ({run}, {tokens/1e6:,.1f}M tokens)")
    base = dict(arm=name, tokens=tokens, params=model.num_params(), ice_params=model.ice_params(),
                kv_bytes_per_token=kv_bytes_per_token(mcfg), seed=cfg["train"].get("seed"),
                train_gflops_per_token=model.flops_per_token(cfg["train"]["seq_len"]) / 1e9)
    variants = [(name, 1.0)] + ([(f"{name}_off", 0.0)] if uses_ice(model) and not a.no_ice_off else [])
    for label, keep in variants:
        res = dict(base, arm=label, **evaluate(model, seq_len, keep))
        for k, v in res.items():
            if not isinstance(v, list):
                print(f"  {k:48s} {v:.4f}" if isinstance(v, float) else f"  {k:48s} {v}")
        all_res[f"{run}#{label}"] = res
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

def table(title, rows):
    cols = list(all_res.values())
    print(f"\n== {title}\n" + "metric".ljust(40) + "".join(r["arm"].rjust(15) for r in cols))
    for label, key in rows:
        vals = [r.get(key) for r in cols]
        if all(v is None for v in vals):
            continue
        print(label.ljust(40) + "".join(
            (f"{v:15.4f}" if isinstance(v, float) and not math.isnan(v) else f"{'-' if v is None else v:>15}")
            for v in vals))


table("cost", [("params", "params"), ("ICE params", "ice_params"), ("KV bytes/token", "kv_bytes_per_token"),
              ("train GFLOPs/token", "train_gflops_per_token"), ("seed", "seed"), ("tokens", "tokens")])
table("held-out loss (lower is better)", [
    (f"{s}/{g}", f"{s}/{g}_loss") for s in ("code", "prose")
    for g in ("all", "other", "copyable", "copy_far", "lk_right", "lk_wrong", "lk_none")])
ref = next((r for r in all_res.values() if r["arm"] == a.reference), None)
for r in all_res.values():
    for s_ in ("code", "prose"):
        if ref is not None:
            r[f"{s_}/follow_excess"] = r[f"{s_}/follow_wrong_lookup"] - ref[f"{s_}/follow_wrong_lookup"]
table("misleading lookups: P(top-1 = wrong lookup token); excess = minus the reference arm's rate",
      [(f"{s}/{m}", f"{s}/{m}") for s in ("code", "prose") for m in ("follow_wrong_lookup", "follow_excess")])


def paired_diff(x, y, n_boot=2000, seed=0):
    """mean(x - y) and a paired bootstrap 95% interval (x, y: 0/1 per item, same items)."""
    d = np.asarray(x, float) - np.asarray(y, float)
    if len(d) == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    boots = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(1)
    return float(d.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


if tok is not None:
    for r in all_res.values():
        for cat, v in summarize(r, "dec_acc").items():
            r[f"cr/{cat}"] = v
        for cat, v in summarize(r, "dec_lookup_wrong_frac").items():
            r[f"cr_wrongfrac/{cat}"] = v
        for cat, v in summarize(r, "dec_follow_wrong_lookup").items():
            r[f"cr_follow/{cat}"] = v
    table("CodeRecall: accuracy on the deciding digit (mean over gaps)", [(c, f"cr/{c}") for c in CATEGORIES])
    table("CodeRecall: share of items where the lookup is wrong on the deciding digit",
          [(c, f"cr_wrongfrac/{c}") for c in CATEGORIES])
    table("CodeRecall: follow rate on wrong lookups (deciding digit)", [(c, f"cr_follow/{c}") for c in CATEGORIES])
    if ref is not None:
        print(f"\n== CodeRecall: deciding-digit accuracy minus {a.reference} [paired bootstrap 95% CI], items pooled over gaps")
        print("category".ljust(16) + "".join(r["arm"].rjust(26) for r in all_res.values()))
        for cat in CATEGORIES:
            cells = []
            for r in all_res.values():
                xs, ys = [], []
                for k, v in r.items():
                    if k.startswith(f"coderecall/{cat}@") and k.endswith("/items_dec") \
                            and len(ref.get(k, [])) == len(v):
                        xs += v
                        ys += ref[k]
                m, lo, hi = paired_diff(xs, ys)
                cells.append(f"{m:+.3f} [{lo:+.3f},{hi:+.3f}]")
            print(cat.ljust(16) + "".join(c.rjust(26) for c in cells))
table("synthetic / needle tasks", [(k, k) for k in next(iter(all_res.values()))
                                   if k.endswith(("/acc", "/em")) and not k.startswith("coderecall/")])
with open(a.out, "w") as f:
    json.dump(all_res, f, indent=2)
print(f"\nsaved {a.out}")


In [ ]:
%%writefile ice_lm/scripts/benchmark.py
#!/usr/bin/env python
"""Inference cost per arm: prefill time, decode speed, and memory, including the ICE lookup and its tables.

    python scripts/benchmark.py --preset 0.6b --arms A B C F --contexts 4096 16384 32768 --data_dir data
    python scripts/benchmark.py --runs runs/0.6b_A runs/0.6b_C ...      # trained weights instead of random ones

Speed and memory do not depend on the weights, so random weights are fine; with --data_dir the prompt and
the decoded tokens are real held-out code (teacher-forced), so ICE hit rates are realistic and identical
across arms. Every arm runs the same loop: ICE lookup (CPU, ICE arms only) -> forward -> argmax -> .item().

Memory model reported per arm (cache memory only: activations / workspace are NOT included, so the max batch
is an upper bound; the measured peak memory per run is printed next to it):
    M(batch, T) = weights + batch * (fixed + per_token * T)
    fixed     = GDN recurrent state (fp32) + conv state (measured) + ICE order-1 table (modelled)
    per_token = KV bytes/token (measured) + ICE bytes/token (compact layout modelled from the distinct n-grams
                counted at the largest context, see ice_lm/inference.py; random tokens = worst case)
Prefill time is reported for the model and for the ICE prompt processing separately (the ICE index is
Python here; a C/GPU implementation would be much faster).
"""
import argparse
import json
import math
import os
import statistics
import sys
import time

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import numpy as np  # noqa: E402
import torch  # noqa: E402

from ice_lm.config import build_configs, model_config_from_dict  # noqa: E402
from ice_lm.data import load_meta, open_tokens  # noqa: E402
from ice_lm.inference import FEATURE_KEYS_1, FEATURE_KEYS_N, ICEIndex, features_to_tensors, \
    step_features_to_tensors  # noqa: E402
from ice_lm.model import ICELM, kv_bytes_per_token, layer_plan  # noqa: E402
from ice_lm.utils import pick_precision  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--preset", default="0.6b")
ap.add_argument("--arms", nargs="+", default=["A", "B", "C", "F"])
ap.add_argument("--runs", nargs="*", default=None, help="trained run folders (override --preset/--arms)")
ap.add_argument("--contexts", type=int, nargs="+", default=[4096, 16384, 32768])
ap.add_argument("--batch", type=int, default=1)
ap.add_argument("--decode_steps", type=int, default=64)
ap.add_argument("--warmup_context", type=int, default=1024, help="untimed run per arm first (kernels, caches)")
ap.add_argument("--prefill_chunk", type=int, default=4096)
ap.add_argument("--data_dir", default=None)
ap.add_argument("--precision", default="auto")
ap.add_argument("--budgets_gb", type=float, nargs="+", default=[16, 24, 80])
ap.add_argument("--out", default="benchmark.json")
a = ap.parse_args()
a.contexts = sorted(a.contexts)          # the memory model uses the largest context

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype, _ = pick_precision(a.precision, device)
dtype = amp_dtype or torch.float32
vocab = 32000
eos = None
stream = None
if a.data_dir:
    meta = load_meta(a.data_dir)
    vocab, eos = meta["vocab_size"], meta["eos_id"]
    stream = np.asarray(open_tokens(a.data_dir, "val_code"))


def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


def mem():
    return torch.cuda.max_memory_allocated() if device.type == "cuda" else 0


def tokens_for(B, T, seed):
    rng = np.random.default_rng(seed)
    if stream is None or len(stream) < T + 1:
        return rng.integers(10, vocab - 10, size=(B, T)).astype(np.int64)
    starts = rng.integers(0, len(stream) - T - 1, size=B)
    return np.stack([stream[s: s + T].astype(np.int64) for s in starts])


def models():
    if a.runs:
        for run in a.runs:
            cfg = json.load(open(os.path.join(run, "config.json")))
            m = ICELM(model_config_from_dict(cfg["model"]))
            path = os.path.join(run, "model_final.pt")
            path = path if os.path.exists(path) else os.path.join(run, "ckpt_latest.pt")
            m.load_state_dict(torch.load(path, map_location="cpu", weights_only=False)["model"])
            yield cfg["train"]["arm"], m
    else:
        for arm in a.arms:
            yield arm, ICELM(build_configs(a.preset, arm, vocab_size=vocab)[0])


def bench(arm, model, T, steps=None):
    cfg = model.cfg
    B, steps = a.batch, max(1, a.decode_steps if steps is None else steps)
    toks = tokens_for(B, T + steps, seed=T)
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    base = mem()
    cache = model.allocate_cache(B, T + steps, device, dtype)
    cache_b = cache.nbytes()
    idx = [ICEIndex(cfg.ice_orders, eos, cfg.vocab_size) for _ in range(B)] if cfg.ice else None

    # ICE prompt processing (CPU, vectorized), timed on its own
    ice_prefill_s, feats = 0.0, None
    if idx:
        t1 = time.perf_counter()
        feats = [ix.prefill(row) for ix, row in zip(idx, toks[:, :T])]
        ice_prefill_s = time.perf_counter() - t1
    # model prefill (chunked, so 32K fits on a 16 GB GPU)
    sync()
    t0 = time.perf_counter()
    for c0 in range(0, T, a.prefill_chunk):
        c1 = min(T, c0 + a.prefill_chunk)
        f = None
        if feats:
            f = features_to_tensors([{**{k: fr[k][:, c0:c1] for k in FEATURE_KEYS_N},
                                      **{k: fr[k][c0:c1] for k in FEATURE_KEYS_1}} for fr in feats], device)
        logits = model.forward_cached(torch.from_numpy(toks[:, c0:c1]).to(device), cache, f)
    logits.argmax(-1).cpu()
    sync()
    prefill_s = time.perf_counter() - t0
    peak_prefill = mem() - base

    # decode, teacher-forced on the real continuation
    step_ms, ice_us = [], []
    for s in range(steps):
        sync()
        t0 = time.perf_counter()
        nxt = toks[:, T + s]
        f = None
        if idx:
            t1 = time.perf_counter()
            f = step_features_to_tensors([ix.step(int(t)) for ix, t in zip(idx, nxt)], device)
            ice_us.append((time.perf_counter() - t1) * 1e6)
        logits = model.forward_cached(torch.from_numpy(nxt[:, None]).to(device), cache, f)
        logits.argmax(-1).cpu()                       # every arm syncs once per step, as in real sampling
        step_ms.append((time.perf_counter() - t0) * 1e3)
    warm = step_ms[min(4, len(step_ms) - 1):]          # drop the first steps (allocator / kernel warmup)
    ms = statistics.median(warm)
    ice_b = idx[0].compact_bytes() if idx else dict(total=0, tokens=0, tables=0, reliability=0)
    return dict(
        arm=arm, context=T, batch=B,
        prefill_s=prefill_s, prefill_tok_s=B * T / prefill_s, ice_prefill_s=ice_prefill_s,
        prefill_tok_s_incl_ice=B * T / (prefill_s + ice_prefill_s),
        decode_ms=ms, decode_tok_s=B * 1e3 / ms,
        # the ICE index here is Python: also report decode speed without its CPU time (a C/GPU index is ~free)
        decode_tok_s_model_only=B * 1e3 / max(1e-6, ms - (statistics.median(ice_us) / 1e3 if ice_us else 0.0)),
        ice_us_per_step_median=statistics.median(ice_us) if ice_us else 0.0,
        ice_us_per_step_p99=float(np.percentile(ice_us, 99)) if ice_us else 0.0,
        kv_bytes=cache_b["kv"], gdn_state_bytes=cache_b["gdn_state"], conv_bytes=cache_b["conv"],
        ice_bytes_per_seq=ice_b["total"], ice_table_bytes=ice_b["tables"],
        ice_distinct_ngrams=idx[0].distinct_ngrams() if idx else [],
        peak_prefill_bytes=peak_prefill, peak_decode_bytes=mem() - base,
    )


results, summary = [], {}
for arm, model in models():
    model = model.to(device=device, dtype=dtype).eval()
    model.prepare_backend(device, dtype)
    cfg = model.cfg
    w_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    print(f"\n### {arm}: {model.num_params()/1e6:.1f}M params ({w_bytes/2**20:.0f} MiB in {dtype}), "
          f"layout {''.join('A' if p['gdn'] == 0 else ('G' if p['attn'] == 0 else 'h') for p in layer_plan(cfg))}")
    rows = []
    for T in [None] + a.contexts:                    # None = warmup run (kernels, caches), not reported
        oom = False
        try:
            r = bench(arm, model, min(a.warmup_context, a.contexts[0]), steps=8) if T is None else bench(arm, model, T)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" not in str(e).lower():
                raise
            oom = True                               # handled outside the except block so its memory is freed
        if oom:
            if device.type == "cuda":
                torch.cuda.empty_cache()
            print(f"  T={T or 'warmup'}: out of memory")
            continue
        if T is None:
            continue
        r["weights_bytes"] = w_bytes
        rows.append(r)
        print(f"  T={T:>6}  prefill {r['prefill_s']:.2f}s ({r['prefill_tok_s']:,.0f} tok/s, ICE {r['ice_prefill_s']:.2f}s)"
              f"  decode {r['decode_tok_s']:,.1f} tok/s ({r['decode_ms']:.1f} ms/step, ICE {r['ice_us_per_step_median']:.0f} us)"
              f"  KV {r['kv_bytes']/2**20:.1f} MiB  GDN {r['gdn_state_bytes']/2**20:.1f} MiB"
              f"  ICE {r['ice_bytes_per_seq']/2**20:.2f} MiB"
              + (f"  peak {r['peak_decode_bytes']/2**20:.0f} MiB" if device.type == "cuda" else ""), flush=True)
    results += rows
    if not rows:
        continue
    # memory model: per-token slope and fixed part per sequence (B=1 terms)
    el = torch.tensor([], dtype=dtype).element_size()
    kv_tok = kv_bytes_per_token(cfg, el)
    last = rows[-1]
    ice_fixed = cfg.vocab_size * 8 if (cfg.ice and 1 in cfg.ice_orders) else 0
    ice_tok = (last["ice_bytes_per_seq"] - ice_fixed) / (last["context"] + max(1, a.decode_steps)) if cfg.ice else 0.0
    fixed = (last["gdn_state_bytes"] + last["conv_bytes"]) / a.batch + ice_fixed
    per_tok = kv_tok + ice_tok
    s = dict(arm=arm, weights_bytes=w_bytes, kv_bytes_per_token=kv_tok, ice_bytes_per_token=ice_tok,
             fixed_bytes_per_seq=fixed, per_token_bytes=per_tok,
             decode_tok_s={r["context"]: r["decode_tok_s"] for r in rows},
             peak_decode_bytes={r["context"]: r["peak_decode_bytes"] for r in rows})
    for T in (32768, 131072):
        s[f"seq_bytes@{T}"] = fixed + per_tok * T
        for gb in a.budgets_gb:
            s[f"max_batch@{T}@{gb:g}GB"] = max(0, math.floor((gb * 2**30 - w_bytes) / (fixed + per_tok * T)))
    summary[arm] = s
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

with open(a.out, "w") as f:
    json.dump(dict(results=results, summary=summary, device=str(device), dtype=str(dtype)), f, indent=2)

print("\n== memory model per sequence (weights shared): M(batch,T) = W + batch*(fixed + per_token*T)"
      "\n   cache memory only (no activations/workspace): max batch is an upper bound")
hdr = ["arm", "W MiB", "fixed MiB", "KV B/tok", "ICE B/tok", "seq@32K MiB", "seq@128K MiB"] + \
      [f"batch@32K/{gb:g}GB" for gb in a.budgets_gb]
print("".join(h.rjust(16) for h in hdr))
for arm, s in summary.items():
    vals = [arm, s["weights_bytes"] / 2**20, s["fixed_bytes_per_seq"] / 2**20, s["kv_bytes_per_token"],
            s["ice_bytes_per_token"], s["seq_bytes@32768"] / 2**20, s["seq_bytes@131072"] / 2**20] + \
           [s[f"max_batch@32768@{gb:g}GB"] for gb in a.budgets_gb]
    print("".join((f"{v:16.1f}" if isinstance(v, float) else str(v).rjust(16)) for v in vals))
print(f"\nsaved {a.out}")


In [ ]:
%%writefile ice_lm/scripts/plot_results.py
#!/usr/bin/env python
"""Presentation figures from scripts/evaluate.py (and optionally scripts/benchmark.py) output.

    python scripts/plot_results.py --results results_140m.json [--benchmark benchmark_140m.json] --out figs

Runs with the same arm name (different seeds) are pooled: points/bars show the mean, error bars the spread
(min-max over seeds) or a 95% Wilson interval over items. Figures:
  frontier.png     code loss vs cache bytes per token (KV + ICE): what each arm pays and gets
  coderecall.png   CodeRecall deciding-digit accuracy per category, ordered from "lookup right" to "lookup wrong"
  harm.png         misleading lookups: follow rate and loss on tokens where the lookup is wrong (two panels)
"""
import argparse
import json
import math
import os
from collections import defaultdict

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

# reference categorical palette (light mode), assigned per arm in a fixed order -- never by rank
ARM_COLORS = {"A": "#2a78d6", "B": "#eb6834", "C": "#1baf7a", "C_notrust": "#eda100", "B_pm": "#e87ba4",
              "F": "#008300", "C_off": "#6250d6"}
OTHER = "#e34948"
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
CATEGORY_ORDER = ["copy_same", "override_same", "conflict_g1", "copy_diff", "conflict_g2", "conflict_g5",
                  "override_diff", "increment", "track"]          # lookup right -> lookup wrong

ap = argparse.ArgumentParser()
ap.add_argument("--results", required=True)
ap.add_argument("--benchmark", default=None, help="benchmark.json of the SAME preset (adds ICE bytes/token)")
ap.add_argument("--out", default="figs")
a = ap.parse_args()
os.makedirs(a.out, exist_ok=True)

res = json.load(open(a.results))
groups = defaultdict(list)                        # arm label -> list of result dicts (seeds)
for r in res.values():
    groups[r["arm"]].append(r)
arms = sorted(groups, key=lambda s: (list(ARM_COLORS).index(s) if s in ARM_COLORS else 99, s))
color = {arm: ARM_COLORS.get(arm, OTHER) for arm in arms}
bench = json.load(open(a.benchmark))["summary"] if a.benchmark else {}


def style(ax, title, ylabel, xlabel=None):
    ax.set_facecolor(SURFACE)
    ax.set_title(title, color=INK, fontsize=11, loc="left")
    ax.set_ylabel(ylabel, color=INK2, fontsize=9)
    if xlabel:
        ax.set_xlabel(xlabel, color=INK2, fontsize=9)
    ax.tick_params(colors=INK2, labelsize=8, length=0)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)


def vals(arm, key):
    v = [r.get(key) for r in groups[arm]]
    return [x for x in v if isinstance(x, (int, float)) and not math.isnan(x)]


def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


# 1. frontier: quality vs cache bytes per token ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.4, 4.2), facecolor=SURFACE)
for arm in arms:
    if arm.endswith("_off"):
        continue
    loss = vals(arm, "code/all_loss")
    kv = vals(arm, "kv_bytes_per_token")
    if not loss or not kv:
        continue
    x = kv[0] + bench.get(arm, {}).get("ice_bytes_per_token", 0.0)
    y = float(np.mean(loss))
    yerr = [[y - min(loss)], [max(loss) - y]] if len(loss) > 1 else None
    ax.errorbar([x], [y], yerr=yerr, fmt="o", ms=8, color=color[arm], mec=SURFACE, mew=2, ecolor=color[arm],
                elinewidth=2, capsize=0, zorder=3)
    ax.annotate(f"{arm}  ({len(loss)} seed{'s' if len(loss) > 1 else ''})", (x, y), xytext=(8, 4),
                textcoords="offset points", color=INK, fontsize=8)
style(ax, "Code loss vs cache memory per token", "held-out code loss (lower is better)",
      "cache bytes per token: KV" + (" + ICE tables" if bench else "") + "  (less is cheaper)")
fig.tight_layout()
fig.savefig(os.path.join(a.out, "frontier.png"), dpi=200)
plt.close(fig)

# 2. CodeRecall per category ---------------------------------------------------------------------------------
cats = [c for c in CATEGORY_ORDER if any(k.startswith(f"coderecall/{c}@") for r in res.values() for k in r)]
if cats:
    plot_arms = [arm for arm in arms if any(k.endswith("/items_dec") for r in groups[arm] for k in r)]
    fig, ax = plt.subplots(figsize=(max(7.0, 1.0 * len(cats) * max(1, len(plot_arms)) * 0.32 + 2), 4.2),
                           facecolor=SURFACE)
    width = 0.8 / max(1, len(plot_arms))
    for i, arm in enumerate(plot_arms):
        means, lo, hi = [], [], []
        for c in cats:
            items = [v for r in groups[arm] for k, x in r.items()
                     if k.startswith(f"coderecall/{c}@") and k.endswith("/items_dec") for v in x]
            k_, n_ = sum(items), len(items)
            p = k_ / n_ if n_ else float("nan")
            l_, h_ = wilson(k_, n_)
            means.append(p)
            lo.append(p - l_)
            hi.append(h_ - p)
        xs = np.arange(len(cats)) + (i - (len(plot_arms) - 1) / 2) * width
        ax.bar(xs, means, width * 0.92, color=color[arm], label=arm, edgecolor=SURFACE, linewidth=1)
        ax.errorbar(xs, means, yerr=[lo, hi], fmt="none", ecolor=INK2, elinewidth=1, capsize=0)
    ax.set_xticks(np.arange(len(cats)))
    ax.set_xticklabels([c.replace("_", "\n", 1) for c in cats], fontsize=8)
    ax.set_ylim(0, 1.15)
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    n_right = sum(c in ("copy_same", "override_same", "conflict_g1") for c in cats)
    ax.axvline(n_right - 0.5, color=INK2, linewidth=0.8, linestyle=(0, (3, 3)))
    ax.text((n_right - 1) / 2, 1.08, "lookup right", ha="center", color=INK2, fontsize=8)
    ax.text((n_right + len(cats) - 1) / 2, 1.08, "lookup useless or wrong", ha="center", color=INK2, fontsize=8)
    style(ax, "CodeRecall: accuracy on the deciding digit (95% CI)", "accuracy")
    ax.legend(frameon=False, fontsize=8, ncol=min(6, len(plot_arms)), loc="upper right",
              bbox_to_anchor=(1.0, -0.12))
    fig.tight_layout()
    fig.savefig(os.path.join(a.out, "coderecall.png"), dpi=200)
    plt.close(fig)

# 3. harm: follow rate and loss on misleading lookups (two panels, one scale each) ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.8), facecolor=SURFACE)
for ax, key, title, ylabel in (
        (axes[0], "code/follow_wrong_lookup", "Follows a wrong lookup", "P(top-1 = wrong lookup token), code"),
        (axes[1], "code/lk_wrong_loss", "Loss where the lookup is wrong", "code loss on misleading-lookup tokens")):
    shown = [arm for arm in arms if vals(arm, key)]
    ys = [float(np.mean(vals(arm, key))) for arm in shown]
    err = [[y - min(vals(arm, key)) for arm, y in zip(shown, ys)], [max(vals(arm, key)) - y for arm, y in zip(shown, ys)]]
    ax.bar(range(len(shown)), ys, 0.6, color=[color[s] for s in shown], edgecolor=SURFACE, linewidth=1)
    ax.errorbar(range(len(shown)), ys, yerr=err, fmt="none", ecolor=INK2, elinewidth=1, capsize=0)
    for i, y in enumerate(ys):
        ax.text(i, y, f"{y:.2f}", ha="center", va="bottom", fontsize=8, color=INK)
    ax.set_xticks(range(len(shown)))
    ax.set_xticklabels(shown, fontsize=8)
    style(ax, title, ylabel)
fig.tight_layout()
fig.savefig(os.path.join(a.out, "harm.png"), dpi=200)
plt.close(fig)
print(f"wrote {a.out}/frontier.png, coderecall.png, harm.png")


In [ ]:
%%writefile ice_lm/tests/test_ice_lm.py
"""Correctness tests. Run:  python -m pytest -q tests"""
import os
import sys
import tempfile

import numpy as np
import pytest
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from ice_lm.config import ARMS, build_configs, ice_param_count  # noqa: E402
from ice_lm.data import TrainWindows, collate, make_example, make_synthetic_data, split_batch  # noqa: E402
from ice_lm.gdn import gated_delta_rule_chunked, gated_delta_rule_recurrent  # noqa: E402
from ice_lm.ice_features import (DEFAULT_ORDERS, brute_force_successor, compute_ice_features,  # noqa: E402
                                 segments_from_eos)
from ice_lm.inference import ICEIndex, features_to_tensors, step_features_to_tensors  # noqa: E402
from ice_lm.model import ICELM, kv_bytes_per_token, layer_plan  # noqa: E402


# ------------------------------------------------------------------ ICE features
@pytest.mark.parametrize("seed", range(20))
def test_successor_matches_brute_force(seed):
    rng = np.random.default_rng(seed)
    x = rng.integers(0, 4, size=150)
    seg = np.sort(rng.integers(0, 3, size=150)) if seed % 2 else None
    f = compute_ice_features(x, seg)
    for j, n in enumerate(DEFAULT_ORDERS):
        assert (f["ice_succ"][j] == brute_force_successor(x, n, seg)).all(), (seed, n)


def test_feature_details():
    x = np.array([5, 6, 7, 8, 9, 5, 6, 7, 8])
    f = compute_ice_features(x)
    t = 8                                  # last position: "5 6 7 8" seen at positions 0..3
    j4 = DEFAULT_ORDERS.index(4)
    assert f["ice_succ"][j4, t] == 9
    assert f["ice_match_len"][t] == 4      # cannot extend: x[4]=9 vs x[-?] differ
    assert f["ice_s_star"][t] == 9
    assert f["ice_dist"][j4, t] == int(np.floor(np.log2(5))) + 1
    assert (f["ice_succ"][:, :5] == -1).all()   # nothing repeats in the first 5 tokens


def test_segments_block_cross_document_matches():
    eos = 0
    x = np.array([3, 4, 5, eos, 3, 4, 5])
    f = compute_ice_features(x, segments_from_eos(x, eos))
    assert (f["ice_succ"][:, 4:] == -1).all()


def test_agree_and_reliability_details():
    # "a b X ... a b Y ... a b" -> two earlier occurrences of (a b) with different successors: agree = 3
    x = np.array([1, 2, 7, 5, 1, 2, 8, 6, 1, 2])
    f = compute_ice_features(x)
    j2 = DEFAULT_ORDERS.index(2)
    assert f["ice_succ"][j2, 9] == 8 and f["ice_agree"][j2, 9] == 3
    assert f["ice_agree"][j2, 5] == 1                 # only one earlier occurrence
    # order-2 history at t=9: one prediction (t=5 said 7, truth 8) -> 0% right -> bucket 1
    assert f["ice_rel"][j2, 9] == 1
    assert f["ice_rel"][j2, 5] == 0                   # no resolved prediction yet


def _planted_sequence(seed, T=220, vocab=6, eos=0):
    rng = np.random.default_rng(seed)
    x = rng.integers(1, vocab, size=T)
    x[120:170] = x[20:70]                             # a repeat longer than MAX_MATCH
    x[rng.integers(0, T, size=3)] = eos               # document boundaries
    return x


@pytest.mark.parametrize("seed", range(6))
def test_incremental_index_matches_batch_features(seed):
    x = _planted_sequence(seed)
    ref = compute_ice_features(x, segments_from_eos(x, 0))
    idx = ICEIndex(eos_id=0, vocab_size=6)
    got = idx.extend(x[:90].tolist())
    rest = [idx.step(int(t)) for t in x[90:]]
    for k, v in ref.items():
        inc = np.concatenate([got[k], np.array([r[k] for r in rest]).T], axis=-1)
        assert (inc == v).all(), k
    assert idx.compact_bytes()["total"] > 0
    # vectorized prompt processing gives the same features and the same state to continue from
    idx2 = ICEIndex(eos_id=0, vocab_size=6)
    got2 = idx2.prefill(x[:90])
    rest2 = [idx2.step(int(t)) for t in x[90:]]
    for k, v in ref.items():
        inc = np.concatenate([got2[k], np.array([r[k] for r in rest2]).T], axis=-1)
        assert (inc == v).all(), ("prefill", k)
    assert idx2.tables == idx.tables and idx2.compact_bytes() == idx.compact_bytes()   # same memory accounting


def test_causality_of_features():
    rng = np.random.default_rng(0)
    x = rng.integers(0, 5, size=300)
    f1 = compute_ice_features(x)
    x2 = x.copy(); x2[200:] = rng.integers(0, 5, size=100)
    f2 = compute_ice_features(x2)
    for k in f1:
        assert (np.asarray(f1[k])[..., :200] == np.asarray(f2[k])[..., :200]).all(), k


# ------------------------------------------------------------------ GDN kernel
@pytest.mark.parametrize("T,chunk", [(1, 16), (37, 16), (64, 64), (130, 32)])
def test_chunked_matches_recurrent(T, chunk):
    torch.manual_seed(T)
    B, H, K, V = 2, 3, 8, 5
    q = torch.nn.functional.normalize(torch.randn(B, T, H, K), dim=-1)
    k = torch.nn.functional.normalize(torch.randn(B, T, H, K), dim=-1)
    v = torch.randn(B, T, H, V)
    g = -torch.rand(B, T, H) * 0.5
    beta = torch.rand(B, T, H)
    ref = gated_delta_rule_recurrent(q, k, v, g, beta, K ** -0.5)
    out = gated_delta_rule_chunked(q, k, v, g, beta, K ** -0.5, chunk)
    assert torch.allclose(out, ref, atol=1e-5, rtol=1e-4), (out - ref).abs().max()


def test_chunked_gradients():
    torch.manual_seed(0)
    B, T, H, K = 1, 50, 2, 8
    q, k, v = (torch.randn(B, T, H, K, dtype=torch.float64, requires_grad=True) for _ in range(3))
    g = (-torch.rand(B, T, H, dtype=torch.float64) * 0.3).requires_grad_()
    beta = torch.rand(B, T, H, dtype=torch.float64).requires_grad_()
    qn = torch.nn.functional.normalize(q, dim=-1)
    kn = torch.nn.functional.normalize(k, dim=-1)
    a = gated_delta_rule_chunked(qn, kn, v, g, beta, 0.3, 16)
    b = gated_delta_rule_recurrent(qn, kn, v, g, beta, 0.3)
    ga = torch.autograd.grad(a.square().sum(), (q, k, v, g, beta), retain_graph=True)   # qn, kn are shared
    gb = torch.autograd.grad(b.square().sum(), (q, k, v, g, beta))
    for x, y in zip(ga, gb):
        assert torch.allclose(x.float(), y.float(), atol=1e-4, rtol=1e-3)


# ------------------------------------------------------------------ model
def _batch(cfg, B=2, T=64, seed=0):
    rng = np.random.default_rng(seed)
    exs = []
    for _ in range(B):
        w = rng.integers(1, cfg.vocab_size, size=T + 1)
        w[T // 2: T // 2 + 10] = w[3:13]      # create a copy
        ex = make_example(w, None, cfg.ice_orders, cfg.ice)
        ex["ice_keep"] = torch.tensor(1.0)
        exs.append(ex)
    return collate(exs)


@pytest.mark.parametrize("arm", list(ARMS))
def test_forward_backward_all_arms(arm):
    mcfg, _ = build_configs("smoke", arm, vocab_size=300)
    model = ICELM(mcfg)
    x, y, ice, keep, _ = split_batch(_batch(mcfg), "cpu")
    _, loss = model(x, y, ice=ice, ice_keep=keep)
    loss.backward()
    assert torch.isfinite(loss)
    for name, p in model.named_parameters():
        assert p.grad is not None, f"no grad for {name}"


@pytest.mark.parametrize("arm", ["B", "C"])
def test_chunked_training_loss_matches_full(arm):
    """The training path (chunked + recomputed head/loss) must equal the plain full-logits loss and gradients."""
    mcfg, _ = build_configs("smoke", arm, vocab_size=300)
    model = ICELM(mcfg)
    with torch.no_grad():
        for p in model.parameters():
            p.add_(torch.randn_like(p) * 0.01)
    x, y, ice, keep, _ = split_batch(_batch(mcfg, T=100), "cpu")
    model.LOSS_CHUNK = 32                      # 100 tokens -> 4 uneven chunks
    _, loss_c = model(x, y, ice=ice, ice_keep=keep)
    g_c = torch.autograd.grad(loss_c, list(model.parameters()), allow_unused=True)
    logits = model(x, ice=ice, ice_keep=keep)
    loss_f = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
    g_f = torch.autograd.grad(loss_f, list(model.parameters()), allow_unused=True)
    assert torch.allclose(loss_c, loss_f, atol=1e-5)
    for a, b in zip(g_c, g_f):
        assert (a is None) == (b is None)
        if a is not None:
            assert torch.allclose(a, b, atol=1e-5, rtol=1e-4)


@pytest.mark.parametrize("arm", ["C", "C_notrust", "C_in", "C_out", "E"])
def test_ice_param_count(arm):
    mcfg, _ = build_configs("smoke", arm, vocab_size=300)
    assert ice_param_count(mcfg) == ICELM(mcfg).ice_params()


def test_param_matched_B():
    c, _ = build_configs("smoke", "C", vocab_size=300)
    b, _ = build_configs("smoke", "B", vocab_size=300)
    bpm, _ = build_configs("smoke", "B_pm", vocab_size=300)
    n_c, n_b, n_pm = (ICELM(m).num_params() for m in (c, b, bpm))
    assert n_b < n_pm and abs(n_pm - n_c) <= 3 * bpm.d_model * bpm.n_layers


def test_kv_budget_head_layout_matches_B():
    for preset in ("smoke", "140m", "0.6b"):
        b, _ = build_configs(preset, "B")
        d, _ = build_configs(preset, "D")
        a, _ = build_configs(preset, "A")
        assert kv_bytes_per_token(b) == kv_bytes_per_token(d)
        if preset != "smoke":            # 4-layer smoke model: A and B both have one attention layer
            assert kv_bytes_per_token(a) == 2 * kv_bytes_per_token(b)
        assert all(p["gdn"] + p["attn"] == d.n_heads for p in layer_plan(d))


@pytest.mark.parametrize("arm", ["B", "C", "E"])
def test_model_is_causal(arm):
    """Changing future tokens (and recomputing ICE features) must not change past logits."""
    mcfg, _ = build_configs("smoke", arm, vocab_size=300)
    model = ICELM(mcfg).eval()
    with torch.no_grad():
        for p in model.parameters():          # make ICE non-trivial
            p.add_(torch.randn_like(p) * 0.01)
    rng = np.random.default_rng(1)
    w = rng.integers(1, 300, size=97)
    w2 = w.copy(); w2[60:] = rng.integers(1, 300, size=37)
    outs = []
    for seq in (w, w2):
        b = collate([dict(make_example(seq, None, mcfg.ice_orders, mcfg.ice), ice_keep=torch.tensor(1.0))])
        x, _, ice, keep, _ = split_batch(b, "cpu")
        outs.append(model(x, ice=ice, ice_keep=keep))
    assert torch.allclose(outs[0][:, :59], outs[1][:, :59], atol=1e-5)


def test_ice_dropout_disables_ice():
    """keep=0 must give exactly the same logits as the same weights with ICE modules removed."""
    mcfg, _ = build_configs("smoke", "C", vocab_size=300)
    model = ICELM(mcfg).eval()
    with torch.no_grad():
        for p in model.ice_in.parameters():
            p.add_(torch.randn_like(p) * 0.1)
        model.ice_out.w.weight.add_(1.0)
    x, y, ice, _, _ = split_batch(_batch(mcfg), "cpu")
    off = model(x, ice=ice, ice_keep=torch.zeros(x.shape[0]))
    on = model(x, ice=ice, ice_keep=torch.ones(x.shape[0]))
    ice_in, ice_out = model.ice_in, model.ice_out
    model.ice_in = model.ice_out = None
    base = model(x)
    model.ice_in, model.ice_out = ice_in, ice_out
    assert torch.allclose(off, base, atol=1e-5)
    assert not torch.allclose(on, base, atol=1e-3)


def test_train_windows_deterministic():
    with tempfile.TemporaryDirectory() as d:
        make_synthetic_data(d, train_tokens=50_000, val_tokens=5_000)
        a = TrainWindows(d, 64, DEFAULT_ORDERS, True, 0.2, seed=7)
        b = TrainWindows(d, 64, DEFAULT_ORDERS, True, 0.2, seed=7)
        ia, ib = iter(a), iter(b)
        for _ in range(5):
            ea, eb = next(ia), next(ib)
            assert torch.equal(ea["input_ids"], eb["input_ids"])
            assert torch.equal(ea["ice_succ"], eb["ice_succ"])
            assert ea["ice_keep"] == eb["ice_keep"]


def test_train_windows_global_order_and_resume():
    """Rank r / micro-batch m holds samples (m*world + r)*mb + j; resuming at a sample count continues exactly."""
    with tempfile.TemporaryDirectory() as d:
        make_synthetic_data(d, train_tokens=20_000, val_tokens=2_000)
        ref = TrainWindows(d, 32, DEFAULT_ORDERS, False, 0.0, seed=1)
        data = np.memmap(os.path.join(d, "train.bin"), dtype=np.uint16, mode="r")
        want = lambda i: ref.sample(data, i, {})["input_ids"]
        for rank in (0, 1):
            ds = TrainWindows(d, 32, DEFAULT_ORDERS, False, 0.0, seed=1, rank=rank, world=2, micro_batch=3)
            it = iter(ds)
            got = [next(it)["input_ids"] for _ in range(9)]           # local micro-batches 0, 1, 2
            for m in range(3):
                for j in range(3):
                    assert torch.equal(got[m * 3 + j], want((m * 2 + rank) * 3 + j))
        # resume after 2 optimizer micro-batches of both ranks (12 samples): rank 0 continues at micro-batch 2
        ds = TrainWindows(d, 32, DEFAULT_ORDERS, False, 0.0, seed=1, rank=0, world=2, micro_batch=3, start_index=12)
        assert torch.equal(next(iter(ds))["input_ids"], want((2 * 2 + 0) * 3))


def test_train_windows_cover_data_once_per_pass():
    with tempfile.TemporaryDirectory() as d:
        make_synthetic_data(d, train_tokens=20_000, val_tokens=2_000)
        ds = TrainWindows(d, 64, DEFAULT_ORDERS, False, 0.0, seed=3)
        n_win = (len(np.memmap(os.path.join(d, "train.bin"), dtype=np.uint16, mode="r")) - 1) // 64 - 1
        it = iter(ds)
        first = [tuple(next(it)["input_ids"].tolist()) for _ in range(n_win)]
        assert len(set(first)) == n_win              # no window repeats within a pass


@pytest.mark.parametrize("arm", ["A", "B", "C", "D", "E", "F", "C_notrust"])
def test_cached_decoding_matches_full_forward(arm):
    """prefill (37) + chunked prefill (50) + single decode steps == one full forward pass, every arm."""
    eos = 0
    mcfg, _ = build_configs("smoke", arm, vocab_size=64)
    model = ICELM(mcfg).eval()
    with torch.no_grad():
        for p in model.parameters():
            p.add_(torch.randn_like(p) * 0.02)
    rng = np.random.default_rng(3)
    x = rng.integers(1, 64, size=150)
    x[90:140] = x[10:60]
    x[[30, 100]] = eos
    feats = compute_ice_features(x, segments_from_eos(x, eos), mcfg.ice_orders)
    ice = {k: torch.from_numpy(feats[k])[None] for k in feats} if mcfg.ice else None
    with torch.no_grad():
        full = model(torch.from_numpy(x)[None], ice=ice)                   # [1,T,V]
        cache = model.allocate_cache(1, 160, "cpu", torch.float32)
        idx = ICEIndex(mcfg.ice_orders, eos_id=eos, vocab_size=64)
        outs = []
        for a, b in ((0, 37), (37, 87)):
            f = features_to_tensors([idx.extend(x[a:b].tolist())], "cpu") if mcfg.ice else None
            outs.append(model.forward_cached(torch.from_numpy(x[a:b])[None], cache, f, last_only=False))
        for t in range(87, 150):
            f = step_features_to_tensors([idx.step(int(x[t]))], "cpu") if mcfg.ice else None
            outs.append(model.forward_cached(torch.from_numpy(x[t:t + 1])[None], cache, f))
    inc = torch.cat(outs, dim=1)
    assert cache.pos == 150
    assert torch.allclose(inc, full, atol=1e-4, rtol=1e-4), (inc - full).abs().max()


@pytest.mark.parametrize("arm", ["B", "E"])
def test_cached_decoding_batch2(arm):
    """Two different sequences decoded together (vectorized prompt prefill + steps) == full forward per row."""
    eos = 0
    mcfg, _ = build_configs("smoke", arm, vocab_size=64)
    model = ICELM(mcfg).eval()
    with torch.no_grad():
        for p in model.parameters():
            p.add_(torch.randn_like(p) * 0.02)
    rng = np.random.default_rng(5)
    xs = rng.integers(1, 64, size=(2, 120))
    xs[:, 70:110] = xs[:, 5:45]
    xs[1, 50] = eos
    fulls = []
    with torch.no_grad():
        for x in xs:
            feats = compute_ice_features(x, segments_from_eos(x, eos), mcfg.ice_orders)
            ice = {k: torch.from_numpy(feats[k])[None] for k in feats} if mcfg.ice else None
            fulls.append(model(torch.from_numpy(x)[None], ice=ice))
        full = torch.cat(fulls)
        cache = model.allocate_cache(2, 120, "cpu", torch.float32)
        idx = [ICEIndex(mcfg.ice_orders, eos_id=eos, vocab_size=64) for _ in range(2)]
        f = features_to_tensors([ix.prefill(x[:64]) for ix, x in zip(idx, xs)], "cpu") if mcfg.ice else None
        outs = [model.forward_cached(torch.from_numpy(xs[:, :64]), cache, f, last_only=False)]
        for t in range(64, 120):
            f = step_features_to_tensors([ix.step(int(x[t])) for ix, x in zip(idx, xs)], "cpu") if mcfg.ice else None
            outs.append(model.forward_cached(torch.from_numpy(xs[:, t:t + 1]), cache, f))
    inc = torch.cat(outs, dim=1)
    assert torch.allclose(inc, full, atol=1e-4, rtol=1e-4), (inc - full).abs().max()


## 4. Tests + GDN backend self-test
Unit tests run on CPU in about a minute. The second command shows which GDN kernel will be used on this GPU.

In [ ]:
!cd ice_lm && python -m pip install -q pytest && python -m pytest -q tests 2>&1 | tail -5
# in a separate process, so this kernel keeps no GPU memory during training
!cd ice_lm && python -c "import torch; from ice_lm.utils import pick_precision; from ice_lm.gdn import resolve_backend; d=torch.device('cuda'); amp,_=pick_precision('auto', d); print('amp dtype:', amp, '| GDN backend:', resolve_backend('auto', d, amp, 128)) if torch.cuda.is_available() else print('no GPU')"

### 4b. Smoke run (optional, ~2 min)
Trains tiny arms B and C on an offline copy-heavy toy corpus. C (with ICE) should reach a clearly lower loss than B.

In [ ]:
!cd ice_lm && python scripts/prepare_data.py --synthetic --out {WORK}/data_synth
for arm in ["B", "C"]:
    smoke_cmd = (f"cd ice_lm && python scripts/train.py --preset smoke --arm {arm} --data_dir {WORK}/data_synth "
                 f"--out_dir {WORK}/smoke_runs/{arm} --total_tokens 600000 --eval_every 100 --log_every 50 --resume false")
    print("=== arm", arm); get_ipython().system(smoke_cmd + " 2>&1 | grep -E 'eval|finished|params'")


## 5. Data
Streams FineWeb-Edu (prose) and codeparrot-clean (Python), 70/30 by tokens, with the TinyLlama (Llama, 32K) tokenizer. Skipped if the data already exists.

In [ ]:
if not os.path.exists(f"{DATA_DIR}/meta.json"):
    !cd ice_lm && python scripts/prepare_data.py --out {DATA_DIR} --train_tokens {TRAIN_DATA_TOKENS}
print(json.load(open(f"{DATA_DIR}/meta.json")))

## 6. Day 0 — the ceiling of ICE (no model)
Coverage = how often an earlier match exists; precision = how often its next token is right. If long matches are rare and imprecise, stop here.

In [ ]:
!cd ice_lm && python scripts/day0_stats.py --data_dir {DATA_DIR} --seq_len {SEQ_LEN} --windows 200 --out {WORK}/day0.json

## 7. Train one arm
Uses all GPUs via `torchrun`. Checkpoints every 30 min and at `MAX_HOURS`; re-running this cell (or a new session with `RESUME_FROM`) continues from the checkpoint.

Memory: with 16 GB T4s, the `0.6b` preset uses micro-batch 1 + activation checkpointing, a chunked loss and Adam states sharded over the GPUs (ZeRO-1); `140m` uses micro-batch 2. Resume with the same number of GPUs to keep the optimizer state.

In [ ]:
import glob, shutil
os.makedirs(RUN_DIR, exist_ok=True)
if RESUME_FROM and not os.path.exists(f"{RUN_DIR}/ckpt_latest.pt"):
    # model checkpoint + per-GPU optimizer shards (ckpt_latest.pt.optim<rank>of<world>) + log
    for f in glob.glob(f"{RESUME_FROM}/ckpt_latest.pt*") + glob.glob(f"{RESUME_FROM}/log.jsonl")             + glob.glob(f"{RESUME_FROM}/config.json") + glob.glob(f"{RESUME_FROM}/model_final.pt"):
        shutil.copy(f, RUN_DIR)
    print("copied checkpoint from", RESUME_FROM, os.listdir(RUN_DIR))
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
micro = {"0.6b": 1 if gpu_mem < 40 else 4, "140m": 2 if gpu_mem < 40 else 8, "smoke": 8}[PRESET]
ckpt = str(GRAD_CKPT).lower() if GRAD_CKPT != "auto" else ("true" if (PRESET == "0.6b" or gpu_mem < 40) else "false")
launcher = f"torchrun --standalone --nproc_per_node={max(1, N_GPU)}" if N_GPU > 1 else "python"
cmd = (f"cd ice_lm && {launcher} scripts/train.py --preset {PRESET} --arm {ARM} --data_dir {DATA_DIR} "
       f"--out_dir {RUN_DIR} --seed {SEED} --seq_len {SEQ_LEN} --micro_batch {micro} "
       + (f"--total_tokens {TOTAL_TOKENS} " if TOTAL_TOKENS else "") +
       f"--grad_checkpointing {ckpt} --max_hours {hours_left():.2f} --num_workers 3")
print(cmd)
!{cmd}

## 8. Evaluate
* held-out loss split by prose/code × copyable/other, and by what the exact lookup says about the next token (right / misleading / none) + how often the model follows a misleading lookup
* **CodeRecall** (identifiers and config values in code): copy with the same / different syntax, conflicting keys, later overrides, and cases where copying is wrong (`increment`)
* repeat / MQAR / phonebook / needles (templated vs paraphrased)
* every ICE arm is also scored with ICE switched off (`C_off`): it should not be worse than B.

In [ ]:
import glob
# every run of this preset (all arms and seeds); add folders from attached inputs to compare runs from other sessions
runs = sorted(d for d in glob.glob(f"{WORK}/runs/{PRESET}_*")
              if os.path.exists(f"{d}/model_final.pt") or os.path.exists(f"{d}/ckpt_latest.pt"))
print(runs)
if not runs:
    print("no trained checkpoint yet - nothing to evaluate")
elif session_left() < 1.0:      # the full evaluation takes ~0.5-1 h per arm on a T4
    print(f"only {session_left():.1f} h left in this session: run the evaluation in a separate session")
else:
    !cd ice_lm && python scripts/evaluate.py --runs {" ".join(runs)} --data_dir {DATA_DIR} --n_task 30 --val_batches 500 --out {WORK}/results_{PRESET}.json

## 9. Inference cost: speed and memory, including the ICE lookup
Prefill and decode speed, KV cache, GDN state and ICE table bytes per arm, plus the memory model `M(batch, T) = W + batch·(fixed + per_token·T)` and how many 32K-token sequences fit in 16/24/80 GB. Random weights are fine for speed/memory; pass `--runs` to use trained ones.

In [ ]:
if session_left() < 0.5:
    print("not enough session time left for the benchmark; run it in a separate session")
else:
    !cd ice_lm && python scripts/benchmark.py --preset {PRESET} --arms A B C F --contexts 4096 16384 32768 --decode_steps 64 --data_dir {DATA_DIR} --out {WORK}/benchmark_{PRESET}.json

## 10. Figures for the presentation
Frontier (code loss vs cache bytes/token), CodeRecall per category (ordered from *lookup right* to *lookup wrong*), and misleading-lookup harm. Seeds of the same arm are pooled.

In [ ]:
from IPython.display import Image, display
if os.path.exists(f"{WORK}/results_{PRESET}.json"):
    bench = f"--benchmark {WORK}/benchmark_{PRESET}.json" if os.path.exists(f"{WORK}/benchmark_{PRESET}.json") else ""
    !cd ice_lm && python scripts/plot_results.py --results {WORK}/results_{PRESET}.json {bench} --out {WORK}/figs
    for name in ("frontier", "coderecall", "harm"):
        if os.path.exists(f"{WORK}/figs/{name}.png"):
            display(Image(f"{WORK}/figs/{name}.png"))
else:
    print("run the evaluation first")

## Continuing across Kaggle sessions
Training stops cleanly at `MAX_HOURS` and saves `ckpt_latest.pt`. Two ways to continue (Kaggle's menu names may
differ slightly):

* **Interactive, with persistence:** in the session settings set *Persistence* → *Files only*. `/kaggle/working`
  (data, checkpoints, logs) is then kept between sessions; just run cells 1–3 and 7 again.
* **Unattended (recommended for long runs):** *Save Version* → *Save & Run All*. Kaggle runs the whole notebook in
  the background (up to its time limit) and stores `/kaggle/working` as the version's output. For the next run:
  *Add Input* → that notebook's output, set `RESUME_FROM = "/kaggle/input/<output>/runs/<PRESET>_<ARM>"` and
  point `DATA_DIR` at the attached `data` folder (so the data is not downloaded again), then save another version.

Either way, training continues from the last checkpoint with the same learning-rate schedule and data order.

**Fair comparison:** run every arm with the same `PRESET`, `TOTAL_TOKENS`, `SEQ_LEN`, `SEED`, number of GPUs
and GPU type, so they see the same data in the same order.